# Hospital Operations -- Real-Time Simulator (Microsoft Fabric)

**SYNTHETIC DEMONSTRATION DATA ONLY -- not derived from or usable for real
clinical decision-making.** Acuity, risk, discharge-readiness and alert
outputs in this notebook are simulated operational signals for demo purposes.

This notebook runs a continuous loop that produces small, believable
operational changes (admissions, transfers, discharges, staffing/equipment
status changes, alerts) and writes them to the `ops_*` tables created by
`Hospital_Operations_Setup.ipynb`. It never modifies clinical tables owned by
the Healthcare Data Generator.

## Prerequisites
Run `Hospital_Operations_Setup.ipynb` at least once first.

## Controlling a running simulator
This loop reads `ops_simulation_control.requested_state` every iteration:
- `RUN` (default) -- keep going
- `PAUSE` -- heartbeat and wait, without ending the run
- `STOP` -- exit cleanly, mark the run `STOPPED_BY_USER`

From another notebook cell (or a separate notebook connected to the same
database), call:
```python
set_requested_state(SINGLE_ENGINE, 'Hospital_Operations_Realtime_Simulator', 'PAUSE')  # or 'RUN' / 'STOP'
```
Interrupting this cell (Fabric "Cancel" / Jupyter "Interrupt Kernel") also
triggers a graceful shutdown -- a final checkpoint is saved and the run is
marked `STOPPED_BY_USER` instead of being left `RUNNING` forever.

## Scenarios
`NORMAL_OPERATIONS`, `FRIDAY_ED_SURGE`, `WINTER_RESPIRATORY_SURGE`,
`ICU_CAPACITY_CRISIS`, `STAFFING_SHORTAGE`, `EQUIPMENT_FAILURE`,
`DISCHARGE_BOTTLENECK` -- set via `SCENARIO_NAME` (config cell) or the
`HOSPITAL_OPS_SCENARIO` env var. Scenarios are configuration profiles (tunable
probabilities/thresholds), not separate code paths.


In [ ]:
# ============================================================================
# CELL: Configuration & Environment Setup
# ============================================================================
import importlib
import subprocess
import sys


def _ensure_package(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f'Installing missing package: {pip_name}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', pip_name], check=False)


for _pip_name, _import_name in [
    ('faker', 'faker'),
    ('pyodbc', 'pyodbc'),
    ('python-dotenv', 'dotenv'),
    ('sqlalchemy', 'sqlalchemy'),
]:
    _ensure_package(_pip_name, _import_name)

import os
import json
import math
import random
import socket
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd() / '.env')
except Exception:
    pass

try:
    import notebookutils  # noqa: F401
    RUNNING_IN_FABRIC = True
except ImportError:
    RUNNING_IN_FABRIC = False

FABRIC_SERVER = os.getenv('FABRIC_SERVER', '5mx5ymqwo74ezmng6awpg76wx4-f2smxdwfuzzenbkp2vylj5c5ca.database.fabric.microsoft.com')
FABRIC_DB = os.getenv('FABRIC_DB', 'Healthcare ODS-63ba7f40-a784-49f0-ae76-387233bc2616')

CONNECTION_STRING = os.getenv('CONNECTION_STRING') or os.getenv('FABRIC_CONNECTION_STRING')
if not CONNECTION_STRING:
    if RUNNING_IN_FABRIC:
        CONNECTION_STRING = (
            f'Driver={{ODBC Driver 18 for SQL Server}};Server=tcp:{FABRIC_SERVER},1433;'
            f'Database={FABRIC_DB};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
        )
    else:
        CONNECTION_STRING = (
            f'Driver={{ODBC Driver 18 for SQL Server}};Server=tcp:{FABRIC_SERVER},1433;'
            f'Database={FABRIC_DB};Authentication=ActiveDirectoryInteractive;'
            f'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
        )

FABRIC_POOL_RECYCLE_SECONDS = max(3600, int(os.getenv('FABRIC_POOL_RECYCLE_SECONDS', '43200')))

# ---------------------------------------------------------------------------
# Hospital Operations parameters -- override via env vars (HOSPITAL_OPS_*) or
# by editing this cell directly. Zero for MAX_RUNTIME_MINUTES/MAX_ITERATIONS
# means unlimited (subject to STOP via ops_simulation_control or a manual
# interrupt of this cell).
# ---------------------------------------------------------------------------
SIMULATOR_NAME = 'Hospital_Operations_Realtime_Simulator'
SCENARIO_NAME = os.getenv('HOSPITAL_OPS_SCENARIO', 'NORMAL_OPERATIONS')
UPDATE_INTERVAL_SECONDS = int(os.getenv('HOSPITAL_OPS_UPDATE_INTERVAL_SECONDS', '10'))
SPEED_MULTIPLIER = float(os.getenv('HOSPITAL_OPS_SPEED_MULTIPLIER', '60'))
RANDOM_SEED = int(os.getenv('HOSPITAL_OPS_RANDOM_SEED', '42'))
MAX_RUNTIME_MINUTES = int(os.getenv('HOSPITAL_OPS_MAX_RUNTIME_MINUTES', '0'))
MAX_ITERATIONS = int(os.getenv('HOSPITAL_OPS_MAX_ITERATIONS', '0'))
BATCH_SIZE = int(os.getenv('HOSPITAL_OPS_BATCH_SIZE', '10'))
SELECTED_HOSPITAL_ID = int(os.getenv('HOSPITAL_OPS_SELECTED_HOSPITAL_ID')) if os.getenv('HOSPITAL_OPS_SELECTED_HOSPITAL_ID') else None
RESET_CURRENT_STATE = os.getenv('HOSPITAL_OPS_RESET_CURRENT_STATE', 'false').strip().lower() in ('1', 'true', 'yes', 'on')
RESUME_FROM_CHECKPOINT = os.getenv('HOSPITAL_OPS_RESUME_FROM_CHECKPOINT', 'true').strip().lower() in ('1', 'true', 'yes', 'on')
ENABLE_PATIENT_FLOW = os.getenv('HOSPITAL_OPS_ENABLE_PATIENT_FLOW', 'true').strip().lower() in ('1', 'true', 'yes', 'on')
ENABLE_STAFFING_EVENTS = os.getenv('HOSPITAL_OPS_ENABLE_STAFFING_EVENTS', 'true').strip().lower() in ('1', 'true', 'yes', 'on')
ENABLE_EQUIPMENT_EVENTS = os.getenv('HOSPITAL_OPS_ENABLE_EQUIPMENT_EVENTS', 'true').strip().lower() in ('1', 'true', 'yes', 'on')
ENABLE_ALERTS = os.getenv('HOSPITAL_OPS_ENABLE_ALERTS', 'true').strip().lower() in ('1', 'true', 'yes', 'on')
ENABLE_SNAPSHOT_REFRESH = os.getenv('HOSPITAL_OPS_ENABLE_SNAPSHOT_REFRESH', 'true').strip().lower() in ('1', 'true', 'yes', 'on')
LOG_LEVEL = os.getenv('HOSPITAL_OPS_LOG_LEVEL', 'INFO')
HEARTBEAT_INTERVAL_SECONDS = int(os.getenv('HOSPITAL_OPS_HEARTBEAT_INTERVAL_SECONDS', '10'))
CHECKPOINT_INTERVAL_ITERATIONS = int(os.getenv('HOSPITAL_OPS_CHECKPOINT_INTERVAL', '10'))
MAX_CONSECUTIVE_ERRORS = int(os.getenv('HOSPITAL_OPS_MAX_CONSECUTIVE_ERRORS', '5'))

NOTEBOOK_START_TIME = datetime.now()
SINGLE_ENGINE = None  # initialized in the "Initialize Database Connection" cell

print('SYNTHETIC DEMONSTRATION DATA ONLY -- not derived from or usable for real clinical decision-making.')
print(f'Start time: {NOTEBOOK_START_TIME.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Running in Fabric: {RUNNING_IN_FABRIC}')
print(f'Scenario: {SCENARIO_NAME} | seed={RANDOM_SEED} | speed_multiplier={SPEED_MULTIPLIER}x')


In [ ]:
# ============================================================================
# CELL: Initialize Database Connection + Generic SQL Helpers
# ============================================================================
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.pool import SingletonThreadPool
import pyodbc
import contextlib


def _get_fabric_access_token():
    for audience in ('pbi', 'https://database.windows.net/'):
        try:
            import notebookutils as _nu
            token = _nu.credentials.getToken(audience)
            if token:
                return token
        except Exception:
            pass
        try:
            import mssparkutils as _msu
            token = _msu.credentials.getToken(audience)
            if token:
                return token
        except Exception:
            pass
    return None


def _pyodbc_creator_with_token(odbc_connect_str, token):
    import struct
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    token_bytes = token.encode('utf-16-le')
    token_struct = struct.pack(f'<I{len(token_bytes)}s', len(token_bytes), token_bytes)

    def creator():
        return pyodbc.connect(odbc_connect_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})

    return creator


def init_connection():
    global SINGLE_ENGINE
    if not CONNECTION_STRING:
        raise RuntimeError('CONNECTION_STRING not configured. Set the env var or edit the config cell.')

    fabric_token = _get_fabric_access_token() if RUNNING_IN_FABRIC else None
    if fabric_token:
        print('Initializing SQLAlchemy engine using a non-interactive Fabric access token...')
        raw_odbc_str = (
            f'Driver={{ODBC Driver 18 for SQL Server}};Server=tcp:{FABRIC_SERVER},1433;'
            f'Database={FABRIC_DB};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
        )
        SINGLE_ENGINE = create_engine(
            'mssql+pyodbc://', creator=_pyodbc_creator_with_token(raw_odbc_str, fabric_token),
            pool_pre_ping=True, pool_recycle=FABRIC_POOL_RECYCLE_SECONDS, poolclass=SingletonThreadPool,
        )
    else:
        print('Initializing SQLAlchemy engine with a single pooled connection...')
        SINGLE_ENGINE = create_engine(
            f'mssql+pyodbc:///?odbc_connect={CONNECTION_STRING}',
            pool_pre_ping=True, pool_recycle=FABRIC_POOL_RECYCLE_SECONDS, poolclass=SingletonThreadPool,
        )

    with SINGLE_ENGINE.connect() as test_conn:
        test_conn.execute(text('SELECT 1'))
    print('  \u2713 Connected to database')

    inspector = inspect(SINGLE_ENGINE)
    tables = inspector.get_table_names()
    print(f'  \u2713 Found {len(tables)} existing tables ({sum(1 for t in tables if t.startswith("ops_"))} already ops_*)')
    return SINGLE_ENGINE


def query_db(sql, engine=None, params=None):
    '''Read-only query helper; returns an empty DataFrame (not an exception) on failure.'''
    engine = engine or SINGLE_ENGINE
    try:
        return pd.read_sql(text(sql), engine, params=params or {})
    except Exception as e:
        print(f'[query_db] error: {e}')
        return pd.DataFrame()


def execute(sql, engine=None, params=None):
    engine = engine or SINGLE_ENGINE
    with engine.begin() as conn:
        conn.execute(text(sql), params or {})


def table_exists(table, engine=None):
    engine = engine or SINGLE_ENGINE
    df = query_db("SELECT 1 AS present FROM sys.tables WHERE name = :t AND schema_id = SCHEMA_ID('dbo')", engine, {'t': table})
    return not df.empty


def get_next_id(table, id_col, engine=None):
    engine = engine or SINGLE_ENGINE
    df = query_db(f'SELECT MAX({id_col}) AS max_id FROM dbo.{table}', engine)
    if df.empty or pd.isna(df.iloc[0]['max_id']):
        return 1
    return int(df.iloc[0]['max_id']) + 1


@contextlib.contextmanager
def transaction(engine=None):
    engine = engine or SINGLE_ENGINE
    with engine.begin() as conn:
        yield conn


def upsert_row(conn, table, key_cols, row):
    '''Upsert a single row into a current-state table via T-SQL MERGE.'''
    all_cols = list(row.keys())
    update_cols = [c for c in all_cols if c not in key_cols]
    src_select = ', '.join(f':{c} AS {c}' for c in all_cols)
    on_clause = ' AND '.join(f'tgt.{c} = src.{c}' for c in key_cols)
    update_clause = ', '.join(f'{c} = src.{c}' for c in update_cols) if update_cols else None
    insert_cols = ', '.join(all_cols)
    insert_vals = ', '.join(f'src.{c}' for c in all_cols)
    match_clause = ('WHEN MATCHED THEN UPDATE SET ' + update_clause) if update_clause else ''
    merge_sql = f'''
    MERGE INTO dbo.{table} AS tgt
    USING (SELECT {src_select}) AS src
    ON {on_clause}
    {match_clause}
    WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals});
    '''
    conn.execute(text(merge_sql), row)


def insert_rows(conn, table, rows):
    count = 0
    for row in rows:
        cols = ', '.join(row.keys())
        vals = ', '.join(f':{c}' for c in row.keys())
        conn.execute(text(f'INSERT INTO dbo.{table} ({cols}) VALUES ({vals})'), row)
        count += 1
    return count


init_connection()


In [ ]:
# ============================================================================
# CELL: Shared constants (staffing roles, equipment plan, state machines, scenarios)
# ============================================================================
from dataclasses import dataclass, field
from typing import Optional

STAFF_ROLES = [
    ('CN', 'Charge Nurse'), ('RN', 'Registered Nurse'), ('PCT', 'Patient Care Technician'),
    ('NP', 'Nurse Practitioner'), ('RT', 'Respiratory Therapist'), ('HOSP', 'Hospitalist'),
    ('ATT', 'Attending Physician'), ('CM', 'Case Manager'), ('TRANS', 'Transporter'),
    ('EVS', 'Environmental Services'), ('BME', 'Biomedical Engineer'),
]
PHYSICIAN_ROLE_CODES = {'HOSP', 'ATT'}

SHIFT_SEED = [
    ('DAY', 'Day', '07:00:00', '15:00:00', False),
    ('EVE', 'Evening', '15:00:00', '23:00:00', False),
    ('NOC', 'Night', '23:00:00', '07:00:00', True),
]

EQUIPMENT_TYPES = [
    'Ventilator', 'Infusion Pump', 'Patient Monitor', 'Portable X-Ray', 'Ultrasound',
    'Dialysis Machine', 'MRI', 'CT Scanner', 'Defibrillator', 'ECMO',
]
CRITICAL_EQUIPMENT_TYPES = {'Ventilator', 'ECMO', 'Defibrillator', 'Dialysis Machine'}

EQUIPMENT_BY_UNIT_TYPE = {
    'Critical Care': [
        ('Ventilator', ('per_bed', 0.6)), ('Patient Monitor', ('per_bed', 1.0)),
        ('Infusion Pump', ('per_bed', 2.0)), ('Defibrillator', 2), ('ECMO', 1),
        ('Dialysis Machine', 1),
    ],
    'Emergency': [
        ('Patient Monitor', ('per_bed', 0.75)), ('Defibrillator', 3),
        ('Portable X-Ray', 2), ('Ultrasound', 1),
    ],
    'Medical/Surgical': [('Patient Monitor', ('per_bed', 0.4)), ('Infusion Pump', ('per_bed', 1.0))],
    'Specialty': [('Infusion Pump', ('per_bed', 0.8)), ('Patient Monitor', ('per_bed', 0.3))],
    'Recovery': [('Patient Monitor', ('per_bed', 0.8)), ('Infusion Pump', ('per_bed', 0.6))],
    'Rehabilitation': [('Patient Monitor', ('per_bed', 0.2))],
    'Observation': [('Patient Monitor', ('per_bed', 0.5)), ('Infusion Pump', ('per_bed', 0.5))],
    'Diagnostic': [('MRI', 1), ('CT Scanner', 1), ('Ultrasound', 2), ('Portable X-Ray', 1)],
    'Medical': [('Infusion Pump', ('per_bed', 0.8)), ('Patient Monitor', ('per_bed', 0.3))],
    'default': [('Patient Monitor', ('per_bed', 0.3))],
}

ROOM_OPERATIONAL_STATUSES = ['Available', 'Occupied', 'Reserved', 'Cleaning', 'Maintenance', 'Closed', 'Isolation']
BED_OPERATIONAL_STATUSES = ['Available', 'Reserved', 'Occupied', 'Discharge Pending', 'Cleaning', 'Maintenance', 'Blocked']
MOVEMENT_TYPES = [
    'Admission', 'Internal Transfer', 'ICU Transfer', 'Step-Down Transfer',
    'Procedure Transfer', 'Discharge', 'Environmental Relocation', 'Maintenance Relocation',
]
ALERT_CATEGORIES = ['Capacity', 'Staffing', 'Patient Flow', 'Equipment', 'Environmental', 'Safety', 'Clinical Operations']
ALERT_SEVERITIES = ['Info', 'Warning', 'Critical']
SCENARIOS = [
    'NORMAL_OPERATIONS', 'FRIDAY_ED_SURGE', 'WINTER_RESPIRATORY_SURGE',
    'ICU_CAPACITY_CRISIS', 'STAFFING_SHORTAGE', 'EQUIPMENT_FAILURE', 'DISCHARGE_BOTTLENECK',
]


@dataclass
class ScenarioProfile:
    name: str
    admission_rate_multiplier: float = 1.0
    ed_boarding_multiplier: float = 1.0
    icu_pressure_multiplier: float = 1.0
    equipment_failure_rate: float = 0.01
    staffing_absence_rate: float = 0.05
    discharge_delay_multiplier: float = 1.0
    icu_occupancy_alert_threshold: float = 0.90
    ed_occupancy_alert_threshold: float = 0.90
    rn_coverage_alert_threshold: float = 0.85


SCENARIO_PROFILES = {
    'NORMAL_OPERATIONS': ScenarioProfile('NORMAL_OPERATIONS'),
    'FRIDAY_ED_SURGE': ScenarioProfile('FRIDAY_ED_SURGE', admission_rate_multiplier=1.4, ed_boarding_multiplier=2.0, ed_occupancy_alert_threshold=0.85),
    'WINTER_RESPIRATORY_SURGE': ScenarioProfile('WINTER_RESPIRATORY_SURGE', admission_rate_multiplier=1.3, icu_pressure_multiplier=1.5, equipment_failure_rate=0.02),
    'ICU_CAPACITY_CRISIS': ScenarioProfile('ICU_CAPACITY_CRISIS', icu_pressure_multiplier=2.2, icu_occupancy_alert_threshold=0.80, discharge_delay_multiplier=1.5),
    'STAFFING_SHORTAGE': ScenarioProfile('STAFFING_SHORTAGE', staffing_absence_rate=0.25, rn_coverage_alert_threshold=0.95),
    'EQUIPMENT_FAILURE': ScenarioProfile('EQUIPMENT_FAILURE', equipment_failure_rate=0.15),
    'DISCHARGE_BOTTLENECK': ScenarioProfile('DISCHARGE_BOTTLENECK', discharge_delay_multiplier=2.5, ed_boarding_multiplier=1.5),
}


def get_scenario_profile(name):
    return SCENARIO_PROFILES.get((name or 'NORMAL_OPERATIONS').upper(), SCENARIO_PROFILES['NORMAL_OPERATIONS'])


def heartbeat_line(**kwargs):
    parts = [f'{k}={v}' for k, v in kwargs.items()]
    return 'HEARTBEAT | ' + ' | '.join(parts)


In [ ]:
SQL_001 = '-- ============================================================================\n-- 001_create_operations_schema.sql\n-- Hospital Operations - schema/namespace decision\n-- ============================================================================\n-- DECISION (see docs/HOSPITAL_OPERATIONS_EXISTING_MODEL_ANALYSIS.md section 6/10):\n-- This project uses an `ops_` TABLE-NAME PREFIX in the default `dbo` schema\n-- instead of a dedicated `hospital_operations` SQL schema. Rationale:\n--   1. The existing Healthcare Data Generator never uses non-dbo schemas.\n--   2. A plain prefix is guaranteed to behave identically across SQL Server,\n--      Azure SQL Database, and Microsoft Fabric SQL Database (Principle #13\n--      in the feature request - "preserve compatibility ... to the extent\n--      supported by the existing repository").\n--   3. It keeps every ops object trivially greppable/droppable (see\n--      008_drop_operations_objects.sql) without relying on schema-transfer\n--      operations that are unverified on Fabric SQL Database.\n--\n-- This script is intentionally a no-op placeholder documenting that decision\n-- and is safe to run on any target. If a dedicated schema is later confirmed\n-- to work end-to-end on the target platform, tables can be migrated with:\n--   CREATE SCHEMA hospital_operations;\n--   ALTER SCHEMA hospital_operations TRANSFER dbo.ops_building;\n--   -- ... repeat per table, then update all view definitions.\n-- ============================================================================\nSELECT \'hospital_operations: using ops_ table-name prefix in dbo schema (see 001_create_operations_schema.sql)\' AS decision;\n'

SQL_002 = "-- ============================================================================\n-- 002_create_reference_tables.sql\n-- Hospital Operations - hierarchy, staffing, and equipment reference tables\n-- Idempotent: safe to run multiple times (IF NOT EXISTS guards).\n-- Does NOT touch any existing clinical table (hospitals, departments, floors,\n-- rooms, beds, etc.) - only reads/references their existing primary keys.\n-- ============================================================================\n\n-- ---------------------------------------------------------------------------\n-- ops_building: NEW physical entity (no building concept exists upstream).\n-- ---------------------------------------------------------------------------\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_building' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_building (\n        building_id         INT NOT NULL PRIMARY KEY,\n        hospital_id         INT NOT NULL,               -- FK -> hospitals.hospital_id\n        building_code        VARCHAR(20) NOT NULL,\n        building_name        VARCHAR(100) NOT NULL,\n        building_type        VARCHAR(50) NOT NULL,\n        campus_label         VARCHAR(100) NULL,\n        latitude             DECIMAL(9,6) NULL,          -- synthetic map coordinate\n        longitude            DECIMAL(9,6) NULL,          -- synthetic map coordinate\n        floor_count           INT NOT NULL DEFAULT 0,\n        active_flag           BIT NOT NULL DEFAULT 1,\n        created_datetime     DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        updated_datetime     DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME()\n    );\n    CREATE INDEX ix_ops_building_hospital ON dbo.ops_building (hospital_id);\nEND\nGO\n\n-- ---------------------------------------------------------------------------\n-- ops_unit: NEW operational-hierarchy entity. Maps 1:1 to an existing\n-- `floors` row via the natural key (hospital_id, floor_number) since `floors`\n-- has no surrogate key. UNIQUE constraint prevents duplicate units on re-run.\n-- ---------------------------------------------------------------------------\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_unit' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_unit (\n        unit_id               INT NOT NULL PRIMARY KEY,\n        hospital_id           INT NOT NULL,              -- FK -> hospitals.hospital_id\n        building_id           INT NOT NULL,              -- FK -> ops_building.building_id\n        source_floor_number   INT NOT NULL,              -- FK (natural) -> floors.floor_number\n        source_department     VARCHAR(100) NULL,          -- denormalized from floors.department\n        department_id         INT NULL,                  -- FK -> departments.department_id (nullable, best-effort match)\n        unit_code             VARCHAR(20) NOT NULL,\n        unit_name             VARCHAR(100) NOT NULL,\n        unit_type             VARCHAR(50) NOT NULL,\n        clinical_specialty    VARCHAR(100) NULL,\n        licensed_bed_count    INT NOT NULL DEFAULT 0,\n        staffed_bed_count     INT NOT NULL DEFAULT 0,\n        nurse_ratio_target    DECIMAL(5,2) NOT NULL DEFAULT 4.0,\n        active_flag           BIT NOT NULL DEFAULT 1,\n        created_datetime      DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        updated_datetime      DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        CONSTRAINT uq_ops_unit_hospital_floor UNIQUE (hospital_id, source_floor_number)\n    );\n    CREATE INDEX ix_ops_unit_hospital ON dbo.ops_unit (hospital_id);\n    CREATE INDEX ix_ops_unit_building ON dbo.ops_unit (building_id);\nEND\nGO\n\n-- ---------------------------------------------------------------------------\n-- ops_room_attribute: 1:1 extension of the EXISTING `rooms` table (no\n-- duplicate room inventory). Adds floor-plan coordinates + capability flags.\n-- ---------------------------------------------------------------------------\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_room_attribute' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_room_attribute (\n        room_id                     INT NOT NULL PRIMARY KEY,   -- FK -> rooms.room_id (1:1)\n        unit_id                     INT NOT NULL,               -- FK -> ops_unit.unit_id\n        room_name                   VARCHAR(100) NULL,\n        isolation_capable_flag      BIT NOT NULL DEFAULT 0,\n        negative_pressure_flag      BIT NOT NULL DEFAULT 0,\n        telemetry_capable_flag      BIT NOT NULL DEFAULT 0,\n        private_room_flag           BIT NOT NULL DEFAULT 0,\n        floor_x                     DECIMAL(9,3) NOT NULL DEFAULT 0,\n        floor_y                     DECIMAL(9,3) NOT NULL DEFAULT 0,\n        width                       DECIMAL(9,3) NOT NULL DEFAULT 1,\n        height                      DECIMAL(9,3) NOT NULL DEFAULT 1,\n        active_flag                 BIT NOT NULL DEFAULT 1,\n        created_datetime            DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        updated_datetime            DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME()\n    );\n    CREATE INDEX ix_ops_room_attribute_unit ON dbo.ops_room_attribute (unit_id);\nEND\nGO\n\n-- ---------------------------------------------------------------------------\n-- ops_shift: NEW reference table of configurable day/evening/night shifts.\n-- ---------------------------------------------------------------------------\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_shift' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_shift (\n        shift_id              INT NOT NULL PRIMARY KEY,\n        shift_code            VARCHAR(10) NOT NULL UNIQUE,\n        shift_name            VARCHAR(50) NOT NULL,\n        start_time            TIME NOT NULL,\n        end_time              TIME NOT NULL,\n        crosses_midnight_flag BIT NOT NULL DEFAULT 0\n    );\nEND\nGO\n\n-- ---------------------------------------------------------------------------\n-- ops_staff: synthetic operational staff. Physician roles reuse a real\n-- doctors.provider_id via existing_provider_id; non-physician roles are\n-- synthetic-only (existing_provider_id IS NULL).\n-- ---------------------------------------------------------------------------\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_staff' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_staff (\n        staff_id              INT NOT NULL PRIMARY KEY,\n        existing_provider_id  INT NULL,                  -- FK -> doctors.provider_id (physician roles only)\n        synthetic_staff_number VARCHAR(20) NOT NULL UNIQUE,\n        display_name           VARCHAR(100) NOT NULL,\n        role_code               VARCHAR(20) NOT NULL,\n        role_name               VARCHAR(50) NOT NULL,\n        credential              VARCHAR(20) NULL,\n        primary_hospital_id     INT NOT NULL,            -- FK -> hospitals.hospital_id\n        primary_unit_id         INT NULL,                -- FK -> ops_unit.unit_id\n        active_flag             BIT NOT NULL DEFAULT 1,\n        created_datetime        DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        updated_datetime        DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME()\n    );\n    CREATE INDEX ix_ops_staff_hospital ON dbo.ops_staff (primary_hospital_id);\n    CREATE INDEX ix_ops_staff_unit ON dbo.ops_staff (primary_unit_id);\nEND\nGO\n\n-- ---------------------------------------------------------------------------\n-- ops_staff_assignment: scheduled/actual staff assignments to a unit/room/\n-- bed/encounter for a given shift.\n-- ---------------------------------------------------------------------------\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_staff_assignment' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_staff_assignment (\n        staff_assignment_id     INT NOT NULL PRIMARY KEY,\n        staff_id                INT NOT NULL,            -- FK -> ops_staff.staff_id\n        hospital_id             INT NOT NULL,            -- FK -> hospitals.hospital_id\n        unit_id                 INT NOT NULL,            -- FK -> ops_unit.unit_id\n        room_id                 INT NULL,                -- FK -> rooms.room_id\n        bed_id                  INT NULL,                -- FK -> beds.bed_id\n        encounter_id            INT NULL,                -- FK -> encounters.encounter_id\n        shift_id                INT NOT NULL,            -- FK -> ops_shift.shift_id\n        assignment_start_datetime DATETIME2 NOT NULL,\n        assignment_end_datetime   DATETIME2 NULL,\n        assignment_role          VARCHAR(50) NOT NULL,\n        patient_load             INT NOT NULL DEFAULT 0,\n        status                   VARCHAR(20) NOT NULL DEFAULT 'Scheduled',\n        source_run_id            INT NULL,                -- FK -> ops_simulation_run.simulation_run_id\n        created_datetime         DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        updated_datetime         DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME()\n    );\n    CREATE INDEX ix_ops_staff_assignment_unit ON dbo.ops_staff_assignment (unit_id);\n    CREATE INDEX ix_ops_staff_assignment_staff ON dbo.ops_staff_assignment (staff_id);\n    CREATE INDEX ix_ops_staff_assignment_status ON dbo.ops_staff_assignment (status);\nEND\nGO\n\n-- ---------------------------------------------------------------------------\n-- ops_equipment: synthetic equipment inventory located at unit/room level.\n-- ---------------------------------------------------------------------------\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_equipment' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_equipment (\n        equipment_id             INT NOT NULL PRIMARY KEY,\n        hospital_id              INT NOT NULL,           -- FK -> hospitals.hospital_id\n        building_id              INT NOT NULL,            -- FK -> ops_building.building_id\n        floor_number              INT NOT NULL,            -- denormalized -> floors.floor_number\n        unit_id                   INT NOT NULL,            -- FK -> ops_unit.unit_id\n        room_id                   INT NULL,                -- FK -> rooms.room_id\n        equipment_type            VARCHAR(50) NOT NULL,\n        manufacturer               VARCHAR(50) NULL,\n        model                       VARCHAR(50) NULL,\n        synthetic_serial_number    VARCHAR(30) NOT NULL UNIQUE,\n        criticality                 VARCHAR(20) NOT NULL DEFAULT 'Standard',\n        installation_date           DATE NULL,\n        maintenance_interval_days   INT NOT NULL DEFAULT 180,\n        active_flag                 BIT NOT NULL DEFAULT 1,\n        created_datetime            DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        updated_datetime            DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME()\n    );\n    CREATE INDEX ix_ops_equipment_unit ON dbo.ops_equipment (unit_id);\n    CREATE INDEX ix_ops_equipment_hospital ON dbo.ops_equipment (hospital_id);\nEND\nGO\n"

SQL_003 = "-- ============================================================================\n-- 003_create_current_state_tables.sql\n-- Hospital Operations - current-state tables (upserted, never truncated).\n-- Idempotent: safe to run multiple times (IF NOT EXISTS guards).\n-- ============================================================================\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_staffing_state' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_staffing_state (\n        unit_id                  INT NOT NULL PRIMARY KEY,   -- FK -> ops_unit.unit_id\n        snapshot_datetime        DATETIME2 NOT NULL,\n        scheduled_rn_count       INT NOT NULL DEFAULT 0,\n        present_rn_count         INT NOT NULL DEFAULT 0,\n        required_rn_count        INT NOT NULL DEFAULT 0,\n        scheduled_pct_count       INT NOT NULL DEFAULT 0,\n        present_pct_count         INT NOT NULL DEFAULT 0,\n        open_shift_count           INT NOT NULL DEFAULT 0,\n        charge_nurse_staff_id      INT NULL,                -- FK -> ops_staff.staff_id\n        staffing_coverage_pct       DECIMAL(5,2) NOT NULL DEFAULT 100.00,\n        staffing_status              VARCHAR(20) NOT NULL DEFAULT 'Normal',\n        simulation_run_id             INT NULL\n    );\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_equipment_state' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_equipment_state (\n        equipment_id            INT NOT NULL PRIMARY KEY,   -- FK -> ops_equipment.equipment_id\n        status                   VARCHAR(20) NOT NULL DEFAULT 'Available',\n        availability_status        VARCHAR(20) NOT NULL DEFAULT 'Available',\n        utilization_status         VARCHAR(20) NOT NULL DEFAULT 'Idle',\n        battery_pct                 DECIMAL(5,2) NULL,\n        temperature                 DECIMAL(6,2) NULL,\n        pressure                    DECIMAL(6,2) NULL,\n        error_code                  VARCHAR(20) NULL,\n        maintenance_due_date         DATE NULL,\n        last_seen_datetime            DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        updated_datetime               DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        simulation_run_id               INT NULL\n    );\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_room_state' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_room_state (\n        room_id                       INT NOT NULL PRIMARY KEY,  -- FK -> rooms.room_id\n        operational_status             VARCHAR(20) NOT NULL DEFAULT 'Available',\n        isolation_status                 VARCHAR(20) NOT NULL DEFAULT 'None',\n        cleaning_status                   VARCHAR(20) NOT NULL DEFAULT 'Clean',\n        maintenance_status                 VARCHAR(20) NOT NULL DEFAULT 'None',\n        current_patient_count               INT NOT NULL DEFAULT 0,\n        available_bed_count                  INT NOT NULL DEFAULT 0,\n        last_cleaned_datetime                  DATETIME2 NULL,\n        next_expected_available_datetime         DATETIME2 NULL,\n        updated_datetime                           DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        simulation_run_id                           INT NULL\n    );\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_bed_state' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_bed_state (\n        bed_id                        INT NOT NULL PRIMARY KEY,  -- FK -> beds.bed_id\n        operational_status              VARCHAR(20) NOT NULL DEFAULT 'Available',\n        occupancy_status                  VARCHAR(20) NOT NULL DEFAULT 'Available',\n        encounter_id                       INT NULL,             -- FK -> encounters.encounter_id\n        patient_id                          INT NULL,            -- FK -> patients.patient_id\n        admission_id                         INT NULL,           -- FK -> admissions.admission_id\n        assigned_datetime                      DATETIME2 NULL,\n        expected_release_datetime                DATETIME2 NULL,\n        cleaning_required_flag                     BIT NOT NULL DEFAULT 0,\n        updated_datetime                             DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        simulation_run_id                             INT NULL\n    );\n    CREATE INDEX ix_ops_bed_state_occupancy ON dbo.ops_bed_state (occupancy_status);\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_discharge_readiness' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_discharge_readiness (\n        encounter_id                 INT NOT NULL PRIMARY KEY,   -- FK -> encounters.encounter_id\n        patient_id                    INT NOT NULL,               -- FK -> patients.patient_id\n        expected_discharge_datetime     DATETIME2 NULL,\n        readiness_status                  VARCHAR(20) NOT NULL DEFAULT 'Not Ready',\n        clinical_ready_flag                 BIT NOT NULL DEFAULT 0,\n        medication_ready_flag                 BIT NOT NULL DEFAULT 0,\n        transport_ready_flag                   BIT NOT NULL DEFAULT 0,\n        destination_ready_flag                   BIT NOT NULL DEFAULT 0,\n        education_complete_flag                    BIT NOT NULL DEFAULT 0,\n        outstanding_barrier_count                    INT NOT NULL DEFAULT 0,\n        primary_barrier                                VARCHAR(50) NULL,\n        updated_datetime                                 DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        simulation_run_id                                 INT NULL\n    );\nEND\nGO\n\n-- ---------------------------------------------------------------------------\n-- ops_operational_alert: mutable `status` field (Open/Acknowledged/Resolved)\n-- but rows are NEVER deleted -- resolved alerts remain for history/reporting.\n-- ---------------------------------------------------------------------------\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_operational_alert' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_operational_alert (\n        alert_id             INT NOT NULL PRIMARY KEY,\n        alert_key             VARCHAR(200) NOT NULL,       -- stable dedup key, e.g. 'CAPACITY:ICU:unit=12'\n        hospital_id            INT NOT NULL,               -- FK -> hospitals.hospital_id\n        building_id             INT NULL,\n        floor_number              INT NULL,\n        unit_id                    INT NULL,\n        room_id                     INT NULL,\n        bed_id                       INT NULL,\n        equipment_id                  INT NULL,\n        encounter_id                   INT NULL,\n        alert_category                  VARCHAR(30) NOT NULL,\n        alert_type                        VARCHAR(50) NOT NULL,\n        severity                            VARCHAR(20) NOT NULL,\n        title                                 VARCHAR(200) NOT NULL,\n        description                            VARCHAR(1000) NULL,\n        status                                   VARCHAR(20) NOT NULL DEFAULT 'Open',\n        opened_datetime                            DATETIME2 NOT NULL,\n        acknowledged_datetime                        DATETIME2 NULL,\n        resolved_datetime                              DATETIME2 NULL,\n        source_metric                                    VARCHAR(50) NULL,\n        source_value                                       DECIMAL(10,2) NULL,\n        threshold_value                                      DECIMAL(10,2) NULL,\n        simulation_run_id                                     INT NULL,\n        created_datetime                                        DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        updated_datetime                                          DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME()\n    );\n    CREATE INDEX ix_ops_alert_status ON dbo.ops_operational_alert (status);\n    CREATE INDEX ix_ops_alert_key ON dbo.ops_operational_alert (alert_key);\n    CREATE INDEX ix_ops_alert_hospital ON dbo.ops_operational_alert (hospital_id);\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_simulation_control' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_simulation_control (\n        control_id                 INT NOT NULL PRIMARY KEY,\n        simulator_name               VARCHAR(100) NOT NULL UNIQUE,\n        requested_state                VARCHAR(10) NOT NULL DEFAULT 'RUN',   -- RUN | PAUSE | STOP\n        update_interval_seconds          INT NOT NULL DEFAULT 10,\n        speed_multiplier                   DECIMAL(9,2) NOT NULL DEFAULT 60.0,\n        random_seed                          INT NULL,\n        scenario_name                          VARCHAR(50) NOT NULL DEFAULT 'NORMAL_OPERATIONS',\n        selected_hospital_id                     INT NULL,\n        last_updated_datetime                      DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        updated_by                                   VARCHAR(100) NULL\n    );\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_simulation_run' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_simulation_run (\n        simulation_run_id                  INT NOT NULL PRIMARY KEY,\n        simulator_name                        VARCHAR(100) NOT NULL,\n        scenario_name                            VARCHAR(50) NOT NULL,\n        random_seed                                INT NULL,\n        started_datetime                             DATETIME2 NOT NULL,\n        ended_datetime                                 DATETIME2 NULL,\n        current_status                                   VARCHAR(20) NOT NULL DEFAULT 'RUNNING',\n        iteration_count                                    INT NOT NULL DEFAULT 0,\n        last_successful_iteration_datetime                   DATETIME2 NULL,\n        error_count                                            INT NOT NULL DEFAULT 0,\n        hostname                                                 VARCHAR(200) NULL,\n        configuration_json                                         NVARCHAR(MAX) NULL\n    );\n    CREATE INDEX ix_ops_simulation_run_status ON dbo.ops_simulation_run (current_status);\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'ops_simulation_checkpoint' AND schema_id = SCHEMA_ID('dbo'))\nBEGIN\n    CREATE TABLE dbo.ops_simulation_checkpoint (\n        simulator_name                 VARCHAR(100) NOT NULL,\n        simulation_run_id                 INT NOT NULL,\n        last_completed_iteration            INT NOT NULL DEFAULT 0,\n        simulated_datetime                    DATETIME2 NULL,\n        random_state                            NVARCHAR(MAX) NULL,\n        checkpoint_datetime                       DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(),\n        checkpoint_json                             NVARCHAR(MAX) NULL,\n        CONSTRAINT pk_ops_simulation_checkpoint PRIMARY KEY (simulator_name, simulation_run_id)\n    );\nEND\nGO\n"

SQL_004 = '-- ============================================================================\n-- 004_create_event_tables.sql\n-- Hospital Operations - append-only historical/event tables.\n-- These tables are ONLY ever inserted into, never updated or truncated by the\n-- real-time simulator (Principle: "Do not use delete-and-reload behavior").\n-- Idempotent: safe to run multiple times (IF NOT EXISTS guards).\n-- ============================================================================\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = \'ops_equipment_event\' AND schema_id = SCHEMA_ID(\'dbo\'))\nBEGIN\n    CREATE TABLE dbo.ops_equipment_event (\n        equipment_event_id      BIGINT NOT NULL PRIMARY KEY,\n        equipment_id              INT NOT NULL,              -- FK -> ops_equipment.equipment_id\n        event_datetime              DATETIME2 NOT NULL,\n        event_type                    VARCHAR(50) NOT NULL,\n        severity                        VARCHAR(20) NOT NULL DEFAULT \'Info\',\n        metric_name                       VARCHAR(50) NULL,\n        metric_value                        DECIMAL(10,2) NULL,\n        metric_unit                          VARCHAR(20) NULL,\n        status_before                          VARCHAR(20) NULL,\n        status_after                             VARCHAR(20) NULL,\n        description                                VARCHAR(500) NULL,\n        simulation_run_id                            INT NULL\n    );\n    CREATE INDEX ix_ops_equipment_event_equip ON dbo.ops_equipment_event (equipment_id, event_datetime);\n    CREATE INDEX ix_ops_equipment_event_run ON dbo.ops_equipment_event (simulation_run_id);\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = \'ops_bed_state_event\' AND schema_id = SCHEMA_ID(\'dbo\'))\nBEGIN\n    CREATE TABLE dbo.ops_bed_state_event (\n        bed_state_event_id         BIGINT NOT NULL PRIMARY KEY,\n        bed_id                       INT NOT NULL,             -- FK -> beds.bed_id\n        event_datetime                 DATETIME2 NOT NULL,\n        event_type                       VARCHAR(50) NOT NULL,\n        status_before                      VARCHAR(20) NULL,\n        status_after                         VARCHAR(20) NULL,\n        encounter_id                           INT NULL,\n        patient_id                               INT NULL,\n        reason                                     VARCHAR(200) NULL,\n        simulation_run_id                            INT NULL\n    );\n    CREATE INDEX ix_ops_bed_state_event_bed ON dbo.ops_bed_state_event (bed_id, event_datetime);\n    CREATE INDEX ix_ops_bed_state_event_run ON dbo.ops_bed_state_event (simulation_run_id);\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = \'ops_patient_movement\' AND schema_id = SCHEMA_ID(\'dbo\'))\nBEGIN\n    CREATE TABLE dbo.ops_patient_movement (\n        movement_id                BIGINT NOT NULL PRIMARY KEY,\n        encounter_id                  INT NOT NULL,           -- FK -> encounters.encounter_id\n        patient_id                      INT NOT NULL,         -- FK -> patients.patient_id\n        hospital_id                       INT NOT NULL,       -- FK -> hospitals.hospital_id\n        from_unit_id                       INT NULL,\n        from_room_id                         INT NULL,\n        from_bed_id                            INT NULL,\n        to_unit_id                               INT NULL,\n        to_room_id                                 INT NULL,\n        to_bed_id                                    INT NULL,\n        requested_datetime                             DATETIME2 NOT NULL,\n        accepted_datetime                                DATETIME2 NULL,\n        started_datetime                                   DATETIME2 NULL,\n        completed_datetime                                   DATETIME2 NULL,\n        movement_type                                          VARCHAR(30) NOT NULL,\n        movement_status                                          VARCHAR(20) NOT NULL DEFAULT \'Requested\',\n        priority                                                   VARCHAR(20) NOT NULL DEFAULT \'Routine\',\n        delay_reason                                                 VARCHAR(200) NULL,\n        simulation_run_id                                              INT NULL\n    );\n    CREATE INDEX ix_ops_patient_movement_encounter ON dbo.ops_patient_movement (encounter_id);\n    CREATE INDEX ix_ops_patient_movement_hospital ON dbo.ops_patient_movement (hospital_id, requested_datetime);\n    CREATE INDEX ix_ops_patient_movement_status ON dbo.ops_patient_movement (movement_status);\nEND\nGO\n\nIF OBJECT_ID(\'dbo.ops_simulated_episode\', \'U\') IS NULL\nBEGIN\n    CREATE TABLE dbo.ops_simulated_episode (\n        encounter_id       INT NOT NULL PRIMARY KEY,\n        admission_id       INT NOT NULL,\n        patient_id         INT NOT NULL,\n        simulation_run_id  INT NOT NULL,\n        created_datetime   DATETIME2 NOT NULL\n    );\nEND\nGO\n\n-- Length-of-stay planning columns. Added separately and idempotently so an\n-- operations schema deployed before benchmark-driven length of stay upgrades\n-- in place, without re-running the full setup notebook.\nIF COL_LENGTH(\'dbo.ops_simulated_episode\', \'icd_family\') IS NULL\n    ALTER TABLE dbo.ops_simulated_episode ADD icd_family VARCHAR(40) NULL;\nGO\nIF COL_LENGTH(\'dbo.ops_simulated_episode\', \'target_los_hours\') IS NULL\n    ALTER TABLE dbo.ops_simulated_episode ADD target_los_hours DECIMAL(9, 2) NULL;\nGO\nIF COL_LENGTH(\'dbo.ops_simulated_episode\', \'ed_dwell_hours\') IS NULL\n    ALTER TABLE dbo.ops_simulated_episode ADD ed_dwell_hours DECIMAL(9, 2) NULL;\nGO\nIF COL_LENGTH(\'dbo.ops_simulated_episode\', \'icu_expected_flag\') IS NULL\n    ALTER TABLE dbo.ops_simulated_episode ADD icu_expected_flag BIT NULL;\nGO\nIF COL_LENGTH(\'dbo.ops_simulated_episode\', \'expected_discharge_datetime\') IS NULL\n    ALTER TABLE dbo.ops_simulated_episode ADD expected_discharge_datetime DATETIME2 NULL;\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = \'ops_alert_event\' AND schema_id = SCHEMA_ID(\'dbo\'))\nBEGIN\n    CREATE TABLE dbo.ops_alert_event (\n        alert_event_id           BIGINT NOT NULL PRIMARY KEY,\n        alert_id                    INT NOT NULL,             -- FK -> ops_operational_alert.alert_id\n        event_datetime                 DATETIME2 NOT NULL,\n        event_type                       VARCHAR(30) NOT NULL,\n        status_before                       VARCHAR(20) NULL,\n        status_after                          VARCHAR(20) NULL,\n        description                             VARCHAR(500) NULL,\n        simulation_run_id                         INT NULL\n    );\n    CREATE INDEX ix_ops_alert_event_alert ON dbo.ops_alert_event (alert_id, event_datetime);\nEND\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = \'ops_simulation_event_log\' AND schema_id = SCHEMA_ID(\'dbo\'))\nBEGIN\n    CREATE TABLE dbo.ops_simulation_event_log (\n        simulation_event_log_id    BIGINT NOT NULL PRIMARY KEY,\n        simulation_run_id             INT NOT NULL,           -- FK -> ops_simulation_run.simulation_run_id\n        iteration_number                 INT NOT NULL,\n        event_datetime                     DATETIME2 NOT NULL,\n        event_category                       VARCHAR(30) NOT NULL,\n        entity_type                            VARCHAR(30) NULL,\n        entity_id                                VARCHAR(50) NULL,\n        action                                      VARCHAR(50) NOT NULL,\n        status                                        VARCHAR(20) NOT NULL DEFAULT \'OK\',\n        message                                          VARCHAR(1000) NULL,\n        diagnostic_json                                    NVARCHAR(MAX) NULL\n    );\n    CREATE INDEX ix_ops_sim_event_log_run ON dbo.ops_simulation_event_log (simulation_run_id, iteration_number);\nEND\nGO\n'

SQL_005 = "-- ============================================================================\n-- 005_create_indexes.sql\n-- Hospital Operations - supplemental indexes for the query patterns the\n-- future Fabric App / Rayfin UI will poll frequently (health-system summary,\n-- hospital summary, floor/room drill-down, active alerts, staffing coverage,\n-- equipment availability, recent event history). Most per-table indexes were\n-- already created inline in 002/003/004; this script adds the remaining\n-- cross-table lookup indexes. Idempotent (guarded by sys.indexes checks).\n-- Deliberately NOT over-indexing append-only event tables beyond what's\n-- needed for entity/timestamp/simulation_run_id lookups.\n-- ============================================================================\n\nIF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name = 'ix_ops_equipment_state_status' AND object_id = OBJECT_ID('dbo.ops_equipment_state'))\n    CREATE INDEX ix_ops_equipment_state_status ON dbo.ops_equipment_state (status, availability_status);\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name = 'ix_ops_room_state_status' AND object_id = OBJECT_ID('dbo.ops_room_state'))\n    CREATE INDEX ix_ops_room_state_status ON dbo.ops_room_state (operational_status);\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name = 'ix_ops_discharge_readiness_status' AND object_id = OBJECT_ID('dbo.ops_discharge_readiness'))\n    CREATE INDEX ix_ops_discharge_readiness_status ON dbo.ops_discharge_readiness (readiness_status);\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name = 'ix_ops_alert_severity' AND object_id = OBJECT_ID('dbo.ops_operational_alert'))\n    CREATE INDEX ix_ops_alert_severity ON dbo.ops_operational_alert (severity, status);\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name = 'ix_ops_equipment_type' AND object_id = OBJECT_ID('dbo.ops_equipment'))\n    CREATE INDEX ix_ops_equipment_type ON dbo.ops_equipment (equipment_type, hospital_id);\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name = 'ix_ops_staff_assignment_bed' AND object_id = OBJECT_ID('dbo.ops_staff_assignment'))\n    CREATE INDEX ix_ops_staff_assignment_bed ON dbo.ops_staff_assignment (bed_id) WHERE bed_id IS NOT NULL;\nGO\n\nIF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name = 'ix_ops_patient_movement_run' AND object_id = OBJECT_ID('dbo.ops_patient_movement'))\n    CREATE INDEX ix_ops_patient_movement_run ON dbo.ops_patient_movement (simulation_run_id);\nGO\n"

SQL_006 = '-- ============================================================================\n-- 006_create_snapshot_views.sql\n-- Hospital Operations - application snapshot views for the future Fabric App\n-- / Rayfin command-center UI. All views are prefixed vw_ops_ to avoid\n-- colliding with the generator\'s own vw_floor_plan / vw_hospital_status /\n-- etc. Every view is built ONLY from actual existing source tables and the\n-- new ops_* tables (see docs/HOSPITAL_OPERATIONS_EXISTING_MODEL_ANALYSIS.md\n-- section 8, source-to-target mapping). CREATE OR ALTER is used so this\n-- script is idempotent/re-runnable, matching the generator\'s own convention.\n-- Upper-level views (health-system/hospital/building/floor/unit) intentionally\n-- expose ONLY counts/aggregates -- never patient names -- per the demo-safety\n-- requirement ("avoid exposing unnecessary synthetic PII at upper navigation\n-- levels").\n-- ============================================================================\n\nCREATE OR ALTER VIEW dbo.vw_ops_bed_snapshot AS\nSELECT\n    b.bed_id, b.room_id, b.hospital_id, b.floor_number, b.room_number, b.bed_position, b.bed_type,\n    b.status AS generator_bed_status,\n    ISNULL(bs.operational_status, \'Available\') AS operational_status,\n    ISNULL(bs.occupancy_status, \'Available\') AS occupancy_status,\n    bs.encounter_id, bs.patient_id, bs.admission_id,\n    bs.assigned_datetime, bs.expected_release_datetime,\n    ISNULL(bs.cleaning_required_flag, 0) AS cleaning_required_flag,\n    u.unit_id, u.unit_name, u.clinical_specialty,\n    bs.updated_datetime, bs.simulation_run_id\nFROM dbo.beds b\nLEFT JOIN dbo.ops_bed_state bs ON b.bed_id = bs.bed_id\nLEFT JOIN dbo.ops_unit u ON u.hospital_id = b.hospital_id AND u.source_floor_number = b.floor_number;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_room_snapshot AS\nSELECT\n    r.room_id, r.hospital_id, r.floor_number, r.room_number, r.bed_count, r.room_type,\n    ra.unit_id, ra.room_name, ra.floor_x, ra.floor_y, ra.width, ra.height,\n    ISNULL(ra.isolation_capable_flag, 0) AS isolation_capable_flag,\n    ISNULL(ra.negative_pressure_flag, 0) AS negative_pressure_flag,\n    ISNULL(ra.telemetry_capable_flag, 0) AS telemetry_capable_flag,\n    ISNULL(ra.private_room_flag, 0) AS private_room_flag,\n    ISNULL(rs.operational_status, \'Available\') AS operational_status,\n    ISNULL(rs.isolation_status, \'None\') AS isolation_status,\n    ISNULL(rs.cleaning_status, \'Clean\') AS cleaning_status,\n    ISNULL(rs.maintenance_status, \'None\') AS maintenance_status,\n    ISNULL(rs.current_patient_count, 0) AS current_patient_count,\n    ISNULL(rs.available_bed_count, r.bed_count) AS available_bed_count,\n    rs.last_cleaned_datetime, rs.next_expected_available_datetime,\n    (SELECT COUNT(*) FROM dbo.ops_operational_alert oa WHERE oa.room_id = r.room_id AND oa.status <> \'Resolved\') AS active_alert_count,\n    rs.updated_datetime, rs.simulation_run_id\nFROM dbo.rooms r\nLEFT JOIN dbo.ops_room_attribute ra ON r.room_id = ra.room_id\nLEFT JOIN dbo.ops_room_state rs ON r.room_id = rs.room_id;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_unit_snapshot AS\nSELECT\n    u.unit_id, u.hospital_id, h.name AS hospital_name, u.building_id, ob.building_name,\n    u.source_floor_number AS floor_number, u.unit_code, u.unit_name, u.unit_type, u.clinical_specialty,\n    u.licensed_bed_count, u.staffed_bed_count,\n    (SELECT COUNT(*) FROM dbo.beds b WHERE b.hospital_id = u.hospital_id AND b.floor_number = u.source_floor_number) AS total_beds,\n    (SELECT COUNT(*) FROM dbo.ops_bed_state bs JOIN dbo.beds b ON bs.bed_id = b.bed_id\n        WHERE b.hospital_id = u.hospital_id AND b.floor_number = u.source_floor_number AND bs.occupancy_status = \'Occupied\') AS occupied_beds,\n    (SELECT COUNT(*) FROM dbo.ops_bed_state bs JOIN dbo.beds b ON bs.bed_id = b.bed_id\n        WHERE b.hospital_id = u.hospital_id AND b.floor_number = u.source_floor_number AND bs.occupancy_status = \'Available\') AS available_beds,\n    (SELECT COUNT(*) FROM dbo.ops_bed_state bs JOIN dbo.beds b ON bs.bed_id = b.bed_id\n        WHERE b.hospital_id = u.hospital_id AND b.floor_number = u.source_floor_number AND bs.occupancy_status = \'Cleaning\') AS cleaning_beds,\n    (SELECT COUNT(*) FROM dbo.ops_bed_state bs JOIN dbo.beds b ON bs.bed_id = b.bed_id\n        WHERE b.hospital_id = u.hospital_id AND b.floor_number = u.source_floor_number AND bs.occupancy_status = \'Blocked\') AS blocked_beds,\n    staff_charge.display_name AS charge_nurse_display_name,\n    ss.required_rn_count, ss.present_rn_count, ss.staffing_coverage_pct, ss.staffing_status,\n    (SELECT COUNT(*) FROM dbo.ops_equipment eq LEFT JOIN dbo.ops_equipment_state es ON eq.equipment_id = es.equipment_id\n        WHERE eq.unit_id = u.unit_id AND ISNULL(es.availability_status, \'Available\') <> \'Available\') AS equipment_unavailable_count,\n    (SELECT COUNT(*) FROM dbo.ops_operational_alert oa WHERE oa.unit_id = u.unit_id AND oa.status <> \'Resolved\') AS active_alert_count,\n    u.active_flag, SYSUTCDATETIME() AS last_refresh_datetime\nFROM dbo.ops_unit u\nJOIN dbo.hospitals h ON u.hospital_id = h.hospital_id\nLEFT JOIN dbo.ops_building ob ON u.building_id = ob.building_id\nLEFT JOIN dbo.ops_staffing_state ss ON u.unit_id = ss.unit_id\nLEFT JOIN dbo.ops_staff staff_charge ON ss.charge_nurse_staff_id = staff_charge.staff_id;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_floor_snapshot AS\nSELECT\n    hospital_id, hospital_name, building_id, building_name, floor_number,\n    COUNT(*) AS unit_count,\n    SUM(total_beds) AS total_beds, SUM(occupied_beds) AS occupied_beds, SUM(available_beds) AS available_beds,\n    SUM(active_alert_count) AS active_alert_count\nFROM dbo.vw_ops_unit_snapshot\nGROUP BY hospital_id, hospital_name, building_id, building_name, floor_number;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_building_snapshot AS\nSELECT\n    ob.building_id, ob.hospital_id, h.name AS hospital_name, ob.building_code, ob.building_name, ob.building_type,\n    ob.latitude, ob.longitude, ob.floor_count,\n    COUNT(DISTINCT u.unit_id) AS unit_count,\n    SUM(u.staffed_bed_count) AS staffed_bed_count,\n    SYSUTCDATETIME() AS last_refresh_datetime\nFROM dbo.ops_building ob\nJOIN dbo.hospitals h ON ob.hospital_id = h.hospital_id\nLEFT JOIN dbo.ops_unit u ON ob.building_id = u.building_id\nGROUP BY ob.building_id, ob.hospital_id, h.name, ob.building_code, ob.building_name, ob.building_type, ob.latitude, ob.longitude, ob.floor_count;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_hospital_snapshot AS\nSELECT\n    h.hospital_id, h.name AS hospital_name, h.specialty AS hospital_type, h.bed_count AS licensed_beds,\n    SUM(u.staffed_bed_count) AS staffed_beds,\n    SUM(CASE WHEN bs.occupancy_status = \'Occupied\' THEN 1 ELSE 0 END) AS occupied_beds,\n    SUM(CASE WHEN bs.occupancy_status = \'Available\' THEN 1 ELSE 0 END) AS available_beds,\n    SUM(CASE WHEN bs.occupancy_status = \'Blocked\' THEN 1 ELSE 0 END) AS blocked_beds,\n    SUM(CASE WHEN bs.occupancy_status = \'Cleaning\' THEN 1 ELSE 0 END) AS cleaning_beds,\n    CASE WHEN SUM(u.staffed_bed_count) > 0\n        THEN ROUND(100.0 * SUM(CASE WHEN bs.occupancy_status = \'Occupied\' THEN 1 ELSE 0 END) / NULLIF(SUM(u.staffed_bed_count), 0), 1)\n        ELSE 0 END AS occupancy_pct,\n    (SELECT COUNT(*) FROM dbo.admissions a WHERE a.hospital_id = h.hospital_id AND a.discharge_datetime IS NULL) AS active_patients,\n    (SELECT COUNT(*) FROM dbo.admissions a JOIN dbo.encounters e ON a.encounter_id = e.encounter_id\n        WHERE a.hospital_id = h.hospital_id AND a.discharge_datetime IS NULL AND e.encounter_type = \'Emergency\') AS ed_patients,\n    (SELECT COUNT(*) FROM dbo.admissions a WHERE a.hospital_id = h.hospital_id AND CAST(a.admit_datetime AS DATE) = CAST(SYSUTCDATETIME() AS DATE)) AS admissions_today,\n    (SELECT COUNT(*) FROM dbo.admissions a WHERE a.hospital_id = h.hospital_id AND CAST(a.discharge_datetime AS DATE) = CAST(SYSUTCDATETIME() AS DATE)) AS discharges_today,\n    (SELECT COUNT(*) FROM dbo.ops_patient_movement pm WHERE pm.hospital_id = h.hospital_id AND pm.movement_status IN (\'Requested\', \'Accepted\', \'InProgress\')) AS transfers_in_progress,\n    (SELECT COUNT(*) FROM dbo.ops_discharge_readiness dr JOIN dbo.admissions a ON dr.encounter_id = a.encounter_id\n        WHERE a.hospital_id = h.hospital_id AND a.discharge_datetime IS NULL AND dr.readiness_status <> \'Ready\') AS pending_discharges,\n    (SELECT AVG(CAST(DATEDIFF(HOUR, a.admit_datetime, ISNULL(a.discharge_datetime, SYSUTCDATETIME())) AS DECIMAL(10,2)) / 24.0)\n        FROM dbo.admissions a WHERE a.hospital_id = h.hospital_id) AS average_length_of_stay_days,\n    ss_agg.staffing_coverage_pct,\n    eq_agg.equipment_available_pct,\n    (SELECT COUNT(*) FROM dbo.ops_operational_alert oa WHERE oa.hospital_id = h.hospital_id AND oa.status <> \'Resolved\') AS active_alert_count,\n    (SELECT COUNT(*) FROM dbo.ops_operational_alert oa WHERE oa.hospital_id = h.hospital_id AND oa.status <> \'Resolved\' AND oa.severity = \'Critical\') AS critical_alert_count,\n    CASE WHEN (SELECT COUNT(*) FROM dbo.ops_operational_alert oa WHERE oa.hospital_id = h.hospital_id AND oa.status <> \'Resolved\' AND oa.severity = \'Critical\') > 0 THEN \'Critical\'\n         WHEN (SELECT COUNT(*) FROM dbo.ops_operational_alert oa WHERE oa.hospital_id = h.hospital_id AND oa.status <> \'Resolved\') > 0 THEN \'Warning\'\n         ELSE \'Normal\' END AS operational_status,\n    SYSUTCDATETIME() AS last_refresh_datetime\nFROM dbo.hospitals h\nLEFT JOIN dbo.ops_unit u ON h.hospital_id = u.hospital_id\nLEFT JOIN dbo.beds bd ON bd.hospital_id = h.hospital_id AND bd.floor_number = u.source_floor_number\nLEFT JOIN dbo.ops_bed_state bs ON bd.bed_id = bs.bed_id\nOUTER APPLY (\n    SELECT AVG(ss.staffing_coverage_pct) AS staffing_coverage_pct\n    FROM dbo.ops_staffing_state ss JOIN dbo.ops_unit u2 ON ss.unit_id = u2.unit_id\n    WHERE u2.hospital_id = h.hospital_id\n) ss_agg\nOUTER APPLY (\n    SELECT CASE WHEN COUNT(*) = 0 THEN 100.0 ELSE\n        ROUND(100.0 * SUM(CASE WHEN ISNULL(es.availability_status, \'Available\') = \'Available\' THEN 1 ELSE 0 END) / COUNT(*), 1) END AS equipment_available_pct\n    FROM dbo.ops_equipment eq LEFT JOIN dbo.ops_equipment_state es ON eq.equipment_id = es.equipment_id\n    WHERE eq.hospital_id = h.hospital_id\n) eq_agg\nGROUP BY h.hospital_id, h.name, h.specialty, h.bed_count, ss_agg.staffing_coverage_pct, eq_agg.equipment_available_pct;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_health_system_snapshot AS\nSELECT\n    COUNT(*) AS hospital_count,\n    SUM(licensed_beds) AS licensed_bed_count,\n    SUM(staffed_beds) AS staffed_bed_count,\n    SUM(occupied_beds) AS occupied_bed_count,\n    SUM(available_beds) AS available_bed_count,\n    CASE WHEN SUM(staffed_beds) > 0 THEN ROUND(100.0 * SUM(occupied_beds) / NULLIF(SUM(staffed_beds), 0), 1) ELSE 0 END AS occupancy_pct,\n    SUM(active_patients) AS active_encounter_count,\n    SUM(admissions_today) AS admissions_today,\n    SUM(discharges_today) AS discharges_today,\n    SUM(pending_discharges) AS pending_discharges,\n    SUM(ed_patients) AS ed_census,\n    (SELECT COUNT(*) FROM dbo.ops_bed_state bs JOIN dbo.ops_unit u ON bs.bed_id IN (SELECT bed_id FROM dbo.beds b WHERE b.floor_number = u.source_floor_number AND b.hospital_id = u.hospital_id)\n        WHERE u.unit_type = \'ICU\' AND bs.occupancy_status = \'Occupied\') AS icu_census,\n    SUM(active_alert_count) AS open_alert_count,\n    SUM(critical_alert_count) AS critical_alert_count,\n    AVG(staffing_coverage_pct) AS staffing_coverage_pct,\n    AVG(equipment_available_pct) AS equipment_available_pct,\n    SYSUTCDATETIME() AS last_refresh_datetime\nFROM dbo.vw_ops_hospital_snapshot;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_patient_operations_snapshot AS\nSELECT\n    p.patient_id, e.encounter_id, a.admission_id,\n    h.hospital_id, h.name AS hospital_name,\n    u.building_id, ob.building_name,\n    u.source_floor_number AS floor_number, u.unit_id, u.unit_name,\n    r.room_id, r.room_number, b.bed_id, b.bed_position,\n    a.admit_datetime,\n    DATEDIFF(HOUR, a.admit_datetime, SYSUTCDATETIME()) / 24.0 AS length_of_stay_days,\n    dr.expected_discharge_datetime, dr.readiness_status AS discharge_readiness,\n    (SELECT TOP 1 pm.movement_status FROM dbo.ops_patient_movement pm WHERE pm.encounter_id = e.encounter_id\n        ORDER BY pm.requested_datetime DESC) AS pending_movement,\n    doc.first_name + \' \' + doc.last_name AS assigned_attending_provider,\n    rn_staff.display_name AS assigned_primary_rn,\n    \'Simulated\' AS operational_acuity_category_label,\n    CASE WHEN pba.los_days IS NOT NULL AND pba.los_days > 10 THEN \'Elevated (simulated)\' ELSE \'Routine (simulated)\' END AS synthetic_operational_risk_indicator,\n    (SELECT COUNT(*) FROM dbo.ops_operational_alert oa WHERE oa.encounter_id = e.encounter_id AND oa.status <> \'Resolved\') AS current_alert_count,\n    SYSUTCDATETIME() AS last_refresh_datetime\nFROM dbo.admissions a\nJOIN dbo.encounters e ON a.encounter_id = e.encounter_id\nJOIN dbo.patients p ON a.patient_id = p.patient_id\nJOIN dbo.hospitals h ON a.hospital_id = h.hospital_id\nLEFT JOIN dbo.doctors doc ON e.provider_id = doc.provider_id\nLEFT JOIN dbo.ops_bed_state bs ON bs.admission_id = a.admission_id\nLEFT JOIN dbo.beds b ON bs.bed_id = b.bed_id\nLEFT JOIN dbo.rooms r ON b.room_id = r.room_id\nLEFT JOIN dbo.ops_unit u ON u.hospital_id = b.hospital_id AND u.source_floor_number = b.floor_number\nLEFT JOIN dbo.ops_building ob ON u.building_id = ob.building_id\nLEFT JOIN dbo.ops_discharge_readiness dr ON dr.encounter_id = e.encounter_id\nLEFT JOIN dbo.patient_bed_assignments pba ON pba.bed_id = b.bed_id AND pba.discharged_datetime IS NULL\nLEFT JOIN dbo.ops_staff_assignment sa ON sa.bed_id = b.bed_id AND sa.status = \'Active\' AND sa.assignment_role = \'Registered Nurse\'\nLEFT JOIN dbo.ops_staff rn_staff ON sa.staff_id = rn_staff.staff_id\nWHERE a.discharge_datetime IS NULL;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_active_alerts AS\nSELECT\n    alert_id, alert_key, hospital_id, building_id, floor_number, unit_id, room_id, bed_id, equipment_id, encounter_id,\n    alert_category, alert_type, severity, title, description, status, opened_datetime, acknowledged_datetime,\n    source_metric, source_value, threshold_value, simulation_run_id,\n    DATEDIFF(MINUTE, opened_datetime, SYSUTCDATETIME()) AS open_duration_minutes\nFROM dbo.ops_operational_alert\nWHERE status <> \'Resolved\';\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_equipment_availability AS\nSELECT\n    eq.equipment_id, eq.hospital_id, eq.unit_id, u.unit_name, eq.room_id, eq.equipment_type, eq.manufacturer, eq.model,\n    eq.criticality, ISNULL(es.status, \'Available\') AS status, ISNULL(es.availability_status, \'Available\') AS availability_status,\n    ISNULL(es.utilization_status, \'Idle\') AS utilization_status, es.battery_pct, es.error_code, es.maintenance_due_date,\n    es.last_seen_datetime, es.simulation_run_id\nFROM dbo.ops_equipment eq\nLEFT JOIN dbo.ops_equipment_state es ON eq.equipment_id = es.equipment_id\nLEFT JOIN dbo.ops_unit u ON eq.unit_id = u.unit_id\nWHERE eq.active_flag = 1;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_staffing_coverage AS\nSELECT\n    u.unit_id, u.hospital_id, h.name AS hospital_name, u.unit_name, u.clinical_specialty,\n    ss.snapshot_datetime, ss.scheduled_rn_count, ss.present_rn_count, ss.required_rn_count,\n    ss.scheduled_pct_count, ss.present_pct_count, ss.open_shift_count,\n    staff_charge.display_name AS charge_nurse_display_name,\n    ss.staffing_coverage_pct, ss.staffing_status, ss.simulation_run_id\nFROM dbo.ops_unit u\nJOIN dbo.hospitals h ON u.hospital_id = h.hospital_id\nLEFT JOIN dbo.ops_staffing_state ss ON u.unit_id = ss.unit_id\nLEFT JOIN dbo.ops_staff staff_charge ON ss.charge_nurse_staff_id = staff_charge.staff_id;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_patient_flow AS\nSELECT\n    pm.movement_id, pm.encounter_id, pm.patient_id, pm.hospital_id,\n    pm.from_unit_id, fu.unit_name AS from_unit_name, pm.from_room_id, pm.from_bed_id,\n    pm.to_unit_id, tu.unit_name AS to_unit_name, pm.to_room_id, pm.to_bed_id,\n    pm.requested_datetime, pm.accepted_datetime, pm.started_datetime, pm.completed_datetime,\n    pm.movement_type, pm.movement_status, pm.priority, pm.delay_reason, pm.simulation_run_id,\n    DATEDIFF(MINUTE, pm.requested_datetime, ISNULL(pm.completed_datetime, SYSUTCDATETIME())) AS elapsed_minutes\nFROM dbo.ops_patient_movement pm\nLEFT JOIN dbo.ops_unit fu ON pm.from_unit_id = fu.unit_id\nLEFT JOIN dbo.ops_unit tu ON pm.to_unit_id = tu.unit_id;\nGO\n\nCREATE OR ALTER VIEW dbo.vw_ops_floor_plan AS\nSELECT\n    h.hospital_id, h.name AS hospital_name, u.building_id, ob.building_name,\n    u.source_floor_number AS floor_number, u.unit_id, u.unit_name, u.clinical_specialty,\n    r.room_id, r.room_number, ra.floor_x, ra.floor_y, ra.width, ra.height,\n    ISNULL(rs.operational_status, \'Available\') AS room_operational_status,\n    b.bed_id, b.bed_position, ISNULL(bs.occupancy_status, \'Available\') AS bed_occupancy_status,\n    bs.patient_id,\n    (SELECT COUNT(*) FROM dbo.ops_operational_alert oa WHERE oa.room_id = r.room_id AND oa.status <> \'Resolved\') AS room_active_alert_count\nFROM dbo.ops_unit u\nJOIN dbo.hospitals h ON u.hospital_id = h.hospital_id\nLEFT JOIN dbo.ops_building ob ON u.building_id = ob.building_id\nLEFT JOIN dbo.rooms r ON r.hospital_id = u.hospital_id AND r.floor_number = u.source_floor_number\nLEFT JOIN dbo.ops_room_attribute ra ON r.room_id = ra.room_id\nLEFT JOIN dbo.ops_room_state rs ON r.room_id = rs.room_id\nLEFT JOIN dbo.beds b ON b.room_id = r.room_id\nLEFT JOIN dbo.ops_bed_state bs ON b.bed_id = bs.bed_id;\nGO\n'

SQL_008_DROP = "-- ============================================================================\n-- 008_drop_operations_objects.sql\n-- Hospital Operations - full teardown. Removes ONLY ops_*/vw_ops_* objects.\n-- Does NOT touch any existing clinical table, view, or the generator's own\n-- floors/rooms/beds/patient_bed_assignments tables.\n-- Safe to run multiple times (IF EXISTS guards). Views dropped before tables;\n-- tables dropped in reverse dependency order.\n-- ============================================================================\n\n-- Views first (they reference the tables below).\nIF OBJECT_ID('dbo.vw_ops_health_system_snapshot', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_health_system_snapshot;\nIF OBJECT_ID('dbo.vw_ops_hospital_snapshot', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_hospital_snapshot;\nIF OBJECT_ID('dbo.vw_ops_building_snapshot', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_building_snapshot;\nIF OBJECT_ID('dbo.vw_ops_floor_snapshot', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_floor_snapshot;\nIF OBJECT_ID('dbo.vw_ops_unit_snapshot', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_unit_snapshot;\nIF OBJECT_ID('dbo.vw_ops_room_snapshot', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_room_snapshot;\nIF OBJECT_ID('dbo.vw_ops_bed_snapshot', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_bed_snapshot;\nIF OBJECT_ID('dbo.vw_ops_patient_operations_snapshot', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_patient_operations_snapshot;\nIF OBJECT_ID('dbo.vw_ops_active_alerts', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_active_alerts;\nIF OBJECT_ID('dbo.vw_ops_equipment_availability', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_equipment_availability;\nIF OBJECT_ID('dbo.vw_ops_staffing_coverage', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_staffing_coverage;\nIF OBJECT_ID('dbo.vw_ops_patient_flow', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_patient_flow;\nIF OBJECT_ID('dbo.vw_ops_floor_plan', 'V') IS NOT NULL DROP VIEW dbo.vw_ops_floor_plan;\nGO\n\n-- Event/history tables (no dependents).\nIF OBJECT_ID('dbo.ops_simulation_event_log', 'U') IS NOT NULL DROP TABLE dbo.ops_simulation_event_log;\nIF OBJECT_ID('dbo.ops_alert_event', 'U') IS NOT NULL DROP TABLE dbo.ops_alert_event;\nIF OBJECT_ID('dbo.ops_patient_movement', 'U') IS NOT NULL DROP TABLE dbo.ops_patient_movement;\nIF OBJECT_ID('dbo.ops_simulated_episode', 'U') IS NOT NULL DROP TABLE dbo.ops_simulated_episode;\nIF OBJECT_ID('dbo.ops_bed_state_event', 'U') IS NOT NULL DROP TABLE dbo.ops_bed_state_event;\nIF OBJECT_ID('dbo.ops_equipment_event', 'U') IS NOT NULL DROP TABLE dbo.ops_equipment_event;\nGO\n\n-- Simulation control/audit tables.\nIF OBJECT_ID('dbo.ops_simulation_checkpoint', 'U') IS NOT NULL DROP TABLE dbo.ops_simulation_checkpoint;\nIF OBJECT_ID('dbo.ops_simulation_run', 'U') IS NOT NULL DROP TABLE dbo.ops_simulation_run;\nIF OBJECT_ID('dbo.ops_simulation_control', 'U') IS NOT NULL DROP TABLE dbo.ops_simulation_control;\nGO\n\n-- Current-state tables.\nIF OBJECT_ID('dbo.ops_operational_alert', 'U') IS NOT NULL DROP TABLE dbo.ops_operational_alert;\nIF OBJECT_ID('dbo.ops_discharge_readiness', 'U') IS NOT NULL DROP TABLE dbo.ops_discharge_readiness;\nIF OBJECT_ID('dbo.ops_bed_state', 'U') IS NOT NULL DROP TABLE dbo.ops_bed_state;\nIF OBJECT_ID('dbo.ops_room_state', 'U') IS NOT NULL DROP TABLE dbo.ops_room_state;\nIF OBJECT_ID('dbo.ops_equipment_state', 'U') IS NOT NULL DROP TABLE dbo.ops_equipment_state;\nIF OBJECT_ID('dbo.ops_staffing_state', 'U') IS NOT NULL DROP TABLE dbo.ops_staffing_state;\nGO\n\n-- Operational/reference tables (staff_assignment before staff/shift; equipment\n-- before unit; room_attribute before unit; unit before building).\nIF OBJECT_ID('dbo.ops_equipment', 'U') IS NOT NULL DROP TABLE dbo.ops_equipment;\nIF OBJECT_ID('dbo.ops_staff_assignment', 'U') IS NOT NULL DROP TABLE dbo.ops_staff_assignment;\nIF OBJECT_ID('dbo.ops_staff', 'U') IS NOT NULL DROP TABLE dbo.ops_staff;\nIF OBJECT_ID('dbo.ops_shift', 'U') IS NOT NULL DROP TABLE dbo.ops_shift;\nIF OBJECT_ID('dbo.ops_room_attribute', 'U') IS NOT NULL DROP TABLE dbo.ops_room_attribute;\nIF OBJECT_ID('dbo.ops_unit', 'U') IS NOT NULL DROP TABLE dbo.ops_unit;\nIF OBJECT_ID('dbo.ops_building', 'U') IS NOT NULL DROP TABLE dbo.ops_building;\nGO\n\nSELECT 'Hospital Operations objects dropped (clinical tables untouched).' AS result;\n"

# ============================================================================
# CELL: Operations Schema DDL + Idempotent Deployment
# The DDL below is embedded verbatim from sql/hospital_operations/*.sql at
# build time (see scripts/build_hospital_operations_notebooks.py) so this
# notebook needs no repo filesystem access to deploy/upgrade the schema.
# Every statement is idempotent (IF NOT EXISTS / CREATE OR ALTER).
# ============================================================================
import re


def _split_batches(sql_text):
    batches = re.split(r'^\s*GO\s*$', sql_text, flags=re.IGNORECASE | re.MULTILINE)
    return [b.strip() for b in batches if b.strip()]


def _run_sql_script(sql_text, engine=None, verbose=True):
    engine = engine or SINGLE_ENGINE
    batches = _split_batches(sql_text)
    with engine.begin() as conn:
        for batch in batches:
            conn.execute(text(batch))
    if verbose:
        print(f'  \u2713 {len(batches)} batch(es) executed')
    return len(batches)


def deploy_schema(engine=None, verbose=True):
    engine = engine or SINGLE_ENGINE
    results = {}
    for name, sql_text in [('001', SQL_001), ('002', SQL_002), ('003', SQL_003), ('004', SQL_004), ('005', SQL_005), ('006', SQL_006)]:
        if verbose:
            print(f'Deploying {name}...')
        results[name] = _run_sql_script(sql_text, engine, verbose)
    return results


def drop_schema(engine=None, verbose=True):
    return _run_sql_script(SQL_008_DROP, engine or SINGLE_ENGINE, verbose)


def list_ops_tables(engine=None):
    df = query_db("SELECT name FROM sys.tables WHERE name LIKE 'ops[_]%' ORDER BY name", engine or SINGLE_ENGINE)
    return df['name'].tolist() if not df.empty else []


def list_ops_views(engine=None):
    df = query_db("SELECT name FROM sys.views WHERE name LIKE 'vw[_]ops[_]%' ORDER BY name", engine or SINGLE_ENGINE)
    return df['name'].tolist() if not df.empty else []


In [ ]:
# ============================================================================
# CELL: Hierarchy Generation (building -> unit -> room attribute extension)
# NEW entities: ops_building, ops_unit. `rooms`/`beds` are REUSED, never duplicated.
# ============================================================================
BUILDING_NAME = 'Main Campus'
BUILDING_TYPE = 'Acute Care'
_LAT_RANGE = (30.0, 45.0)
_LON_RANGE = (-115.0, -75.0)


def generate_buildings(engine=None, seed=None):
    engine = engine or SINGLE_ENGINE
    hospitals = query_db('SELECT hospital_id, name FROM dbo.hospitals', engine)
    existing = query_db('SELECT hospital_id FROM dbo.ops_building', engine)
    existing_ids = set(existing['hospital_id'].tolist()) if not existing.empty else set()
    missing = hospitals[~hospitals['hospital_id'].isin(existing_ids)]
    if not missing.empty:
        next_id = get_next_id('ops_building', 'building_id', engine)
        rng = random.Random(seed)
        with transaction(engine) as conn:
            for _, h in missing.iterrows():
                lat = round(rng.uniform(*_LAT_RANGE), 6)
                lon = round(rng.uniform(*_LON_RANGE), 6)
                upsert_row(conn, 'ops_building', ['building_id'], {
                    'building_id': next_id, 'hospital_id': int(h['hospital_id']),
                    'building_code': f"H{h['hospital_id']}-B1", 'building_name': BUILDING_NAME,
                    'building_type': BUILDING_TYPE, 'campus_label': f"{h['name']} Main Campus",
                    'latitude': lat, 'longitude': lon, 'floor_count': 0, 'active_flag': True,
                    'created_datetime': pd.Timestamp.utcnow().tz_localize(None),
                    'updated_datetime': pd.Timestamp.utcnow().tz_localize(None),
                })
                next_id += 1
    return query_db('SELECT * FROM dbo.ops_building', engine)


def _match_department_id(departments, hospital_id, floor_department):
    if departments.empty or not floor_department:
        return None
    hosp_deps = departments[departments['hospital_id'] == hospital_id]
    if hosp_deps.empty:
        return None
    floor_dep_lower = str(floor_department).lower()
    for _, dep in hosp_deps.iterrows():
        if str(dep['name']).lower() in floor_dep_lower or floor_dep_lower in str(dep['name']).lower():
            return int(dep['department_id'])
    return None


def generate_units(engine=None):
    engine = engine or SINGLE_ENGINE
    floors = query_db('SELECT hospital_id, floor_number, department, bed_type, capacity FROM dbo.floors', engine)
    if floors.empty:
        raise RuntimeError('No rows found in `floors` -- run the Healthcare Data Generator notebook first.')
    departments = query_db('SELECT department_id, hospital_id, name FROM dbo.departments', engine)
    buildings = query_db('SELECT building_id, hospital_id FROM dbo.ops_building', engine)
    existing_units = query_db('SELECT hospital_id, source_floor_number FROM dbo.ops_unit', engine)
    existing_keys = set(zip(existing_units['hospital_id'], existing_units['source_floor_number'])) if not existing_units.empty else set()
    building_by_hospital = dict(zip(buildings['hospital_id'], buildings['building_id'])) if not buildings.empty else {}

    to_create = [f for _, f in floors.iterrows() if (int(f['hospital_id']), int(f['floor_number'])) not in existing_keys]
    if to_create:
        next_id = get_next_id('ops_unit', 'unit_id', engine)
        with transaction(engine) as conn:
            for f in to_create:
                hospital_id = int(f['hospital_id'])
                floor_number = int(f['floor_number'])
                dept_id = _match_department_id(departments, hospital_id, f['department'])
                capacity = int(f['capacity']) if pd.notna(f['capacity']) else 0
                unit_type = str(f['bed_type']) if pd.notna(f['bed_type']) else 'Medical/Surgical'
                upsert_row(conn, 'ops_unit', ['unit_id'], {
                    'unit_id': next_id, 'hospital_id': hospital_id, 'building_id': building_by_hospital.get(hospital_id, 0),
                    'source_floor_number': floor_number,
                    'source_department': str(f['department']) if pd.notna(f['department']) else None,
                    'department_id': dept_id, 'unit_code': f'H{hospital_id}-F{floor_number}',
                    'unit_name': str(f['department']) if pd.notna(f['department']) else f'Floor {floor_number}',
                    'unit_type': unit_type, 'clinical_specialty': str(f['department']) if pd.notna(f['department']) else None,
                    'licensed_bed_count': capacity, 'staffed_bed_count': capacity,
                    'nurse_ratio_target': 2.0 if unit_type == 'Critical Care' else (3.0 if unit_type == 'Emergency' else 4.0),
                    'active_flag': True, 'created_datetime': pd.Timestamp.utcnow().tz_localize(None),
                    'updated_datetime': pd.Timestamp.utcnow().tz_localize(None),
                })
                next_id += 1

    with transaction(engine) as conn:
        counts = query_db('SELECT hospital_id, COUNT(DISTINCT source_floor_number) AS n FROM dbo.ops_unit GROUP BY hospital_id', conn)
        for _, row in counts.iterrows():
            conn.execute(text('UPDATE dbo.ops_building SET floor_count = :n, updated_datetime = SYSUTCDATETIME() WHERE hospital_id = :h'),
                         {'n': int(row['n']), 'h': int(row['hospital_id'])})
    return query_db('SELECT * FROM dbo.ops_unit', engine)


_ISOLATION_PRONE_TYPES = {'Critical Care', 'Emergency'}


def generate_room_attributes(engine=None, seed=None):
    engine = engine or SINGLE_ENGINE
    rooms = query_db('SELECT room_id, hospital_id, floor_number, room_number, bed_count, room_type FROM dbo.rooms', engine)
    if rooms.empty:
        raise RuntimeError('No rows found in `rooms` -- run the Healthcare Data Generator notebook first.')
    units = query_db('SELECT unit_id, hospital_id, source_floor_number, unit_type FROM dbo.ops_unit', engine)
    existing = query_db('SELECT room_id FROM dbo.ops_room_attribute', engine)
    existing_ids = set(existing['room_id'].tolist()) if not existing.empty else set()
    unit_lookup = {(int(u['hospital_id']), int(u['source_floor_number'])): (int(u['unit_id']), u['unit_type']) for _, u in units.iterrows()}

    to_create = rooms[~rooms['room_id'].isin(existing_ids)]
    if not to_create.empty:
        rng = random.Random(seed)
        cols_per_row = 8
        cell_size = 4.0
        with transaction(engine) as conn:
            for _, r in to_create.iterrows():
                key = (int(r['hospital_id']), int(r['floor_number']))
                unit_id, unit_type = unit_lookup.get(key, (0, 'Medical/Surgical'))
                room_number = int(r['room_number'])
                floor_x = ((room_number - 1) % cols_per_row) * cell_size
                floor_y = ((room_number - 1) // cols_per_row) * cell_size
                isolation_prob = 0.30 if unit_type in _ISOLATION_PRONE_TYPES else 0.05
                isolation = rng.random() < isolation_prob
                neg_pressure = isolation and rng.random() < 0.5
                telemetry = unit_type in {'Critical Care', 'Medical/Surgical', 'Emergency'} and rng.random() < 0.7
                upsert_row(conn, 'ops_room_attribute', ['room_id'], {
                    'room_id': int(r['room_id']), 'unit_id': unit_id, 'room_name': f'Room {room_number}',
                    'isolation_capable_flag': isolation, 'negative_pressure_flag': neg_pressure,
                    'telemetry_capable_flag': telemetry, 'private_room_flag': str(r['room_type']) == 'Single',
                    'floor_x': floor_x, 'floor_y': floor_y, 'width': cell_size - 0.5, 'height': cell_size - 0.5,
                    'active_flag': True, 'created_datetime': pd.Timestamp.utcnow().tz_localize(None),
                    'updated_datetime': pd.Timestamp.utcnow().tz_localize(None),
                })
    return query_db('SELECT * FROM dbo.ops_room_attribute', engine)


def generate_hierarchy(engine=None, seed=None):
    buildings = generate_buildings(engine, seed=seed)
    units = generate_units(engine)
    rooms = generate_room_attributes(engine, seed=seed)
    return {'buildings': len(buildings), 'units': len(units), 'room_attributes': len(rooms)}


In [ ]:
# ============================================================================
# CELL: Staffing Generation + Staffing-State Computation
# Physician roles (Hospitalist/Attending) reuse a real doctors.provider_id;
# all other roles are fully synthetic (Faker).
# ============================================================================
from faker import Faker

_NON_PHYSICIAN_ROLE_RATIOS = [('RN', 0.25), ('PCT', 0.15)]


def generate_shifts(engine=None):
    engine = engine or SINGLE_ENGINE
    existing = query_db('SELECT shift_code FROM dbo.ops_shift', engine)
    existing_codes = set(existing['shift_code'].tolist()) if not existing.empty else set()
    to_create = [s for s in SHIFT_SEED if s[0] not in existing_codes]
    if to_create:
        next_id = get_next_id('ops_shift', 'shift_id', engine)
        with transaction(engine) as conn:
            for code, name, start, end, crosses in to_create:
                upsert_row(conn, 'ops_shift', ['shift_id'], {
                    'shift_id': next_id, 'shift_code': code, 'shift_name': name,
                    'start_time': start, 'end_time': end, 'crosses_midnight_flag': crosses,
                })
                next_id += 1
    return query_db('SELECT * FROM dbo.ops_shift', engine)


def _insert_staff(conn, next_id, hospital_id, unit_id, role_code, fake, existing_provider_id=None, display_name=None):
    role_name = dict(STAFF_ROLES)[role_code]
    upsert_row(conn, 'ops_staff', ['staff_id'], {
        'staff_id': next_id, 'existing_provider_id': existing_provider_id,
        'synthetic_staff_number': f'STF-{next_id:06d}', 'display_name': display_name or fake.name(),
        'role_code': role_code, 'role_name': role_name,
        'credential': 'MD' if role_code in PHYSICIAN_ROLE_CODES else ('RN' if role_code in ('CN', 'RN') else None),
        'primary_hospital_id': hospital_id, 'primary_unit_id': unit_id, 'active_flag': True,
        'created_datetime': pd.Timestamp.utcnow().tz_localize(None), 'updated_datetime': pd.Timestamp.utcnow().tz_localize(None),
    })
    return next_id + 1


def generate_staff(engine=None, seed=None):
    engine = engine or SINGLE_ENGINE
    units = query_db('SELECT unit_id, hospital_id, unit_type, staffed_bed_count FROM dbo.ops_unit', engine)
    if units.empty:
        raise RuntimeError('No ops_unit rows found -- run generate_hierarchy() first.')
    doctors = query_db('SELECT provider_id, hospital_id, first_name, last_name FROM dbo.doctors', engine)
    existing = query_db('SELECT DISTINCT primary_unit_id FROM dbo.ops_staff WHERE primary_unit_id IS NOT NULL', engine)
    staffed_unit_ids = set(existing['primary_unit_id'].tolist()) if not existing.empty else set()

    fake = Faker()
    if seed is not None:
        Faker.seed(seed)
        random.seed(seed)

    to_staff = units[~units['unit_id'].isin(staffed_unit_ids)]
    if not to_staff.empty:
        next_id = get_next_id('ops_staff', 'staff_id', engine)
        with transaction(engine) as conn:
            for _, u in to_staff.iterrows():
                unit_id = int(u['unit_id'])
                hospital_id = int(u['hospital_id'])
                bed_count = max(1, int(u['staffed_bed_count']) or 1)
                next_id = _insert_staff(conn, next_id, hospital_id, unit_id, 'CN', fake, existing_provider_id=None)
                for role_code, ratio in _NON_PHYSICIAN_ROLE_RATIOS:
                    n = max(1, round(bed_count * ratio))
                    for _ in range(n):
                        next_id = _insert_staff(conn, next_id, hospital_id, unit_id, role_code, fake, existing_provider_id=None)
                hosp_docs = doctors[doctors['hospital_id'] == hospital_id]
                if not hosp_docs.empty:
                    sample = hosp_docs.sample(n=min(2, len(hosp_docs)), random_state=seed)
                    for i, (_, doc) in enumerate(sample.iterrows()):
                        role_code = 'HOSP' if i == 0 else 'ATT'
                        next_id = _insert_staff(conn, next_id, hospital_id, unit_id, role_code, fake,
                                                 existing_provider_id=int(doc['provider_id']),
                                                 display_name=f"{doc['first_name']} {doc['last_name']}")
    return query_db('SELECT * FROM dbo.ops_staff', engine)


def _current_shift(shifts, as_of):
    if shifts.empty:
        return None
    hour = as_of.hour
    for _, s in shifts.iterrows():
        start_h = int(str(s['start_time']).split(':')[0])
        end_h = int(str(s['end_time']).split(':')[0])
        in_shift = (hour >= start_h or hour < end_h) if s['crosses_midnight_flag'] else (start_h <= hour < end_h)
        if in_shift:
            shift_start = as_of.normalize() + pd.Timedelta(hours=start_h)
            if s['crosses_midnight_flag'] and hour < end_h:
                shift_start -= pd.Timedelta(days=1)
            return int(s['shift_id']), shift_start, None
    first = shifts.iloc[0]
    return int(first['shift_id']), as_of.normalize(), None


def generate_staff_assignments(engine=None, as_of=None, run_id=None):
    engine = engine or SINGLE_ENGINE
    as_of = as_of or pd.Timestamp.utcnow().tz_localize(None)
    staff = query_db("SELECT staff_id, primary_hospital_id, primary_unit_id, role_name FROM dbo.ops_staff WHERE active_flag = 1", engine)
    shifts = query_db('SELECT shift_id, start_time, end_time, crosses_midnight_flag FROM dbo.ops_shift', engine)
    open_assignments = query_db(
        "SELECT DISTINCT staff_id FROM dbo.ops_staff_assignment WHERE status = 'Active' AND assignment_end_datetime IS NULL", engine)
    already_assigned = set(open_assignments['staff_id'].tolist()) if not open_assignments.empty else set()
    current_shift = _current_shift(shifts, as_of)
    to_assign = staff[~staff['staff_id'].isin(already_assigned)]
    if not to_assign.empty and current_shift is not None:
        next_id = get_next_id('ops_staff_assignment', 'staff_assignment_id', engine)
        shift_id, shift_start, shift_end = current_shift
        with transaction(engine) as conn:
            for _, s in to_assign.iterrows():
                upsert_row(conn, 'ops_staff_assignment', ['staff_assignment_id'], {
                    'staff_assignment_id': next_id, 'staff_id': int(s['staff_id']), 'hospital_id': int(s['primary_hospital_id']),
                    'unit_id': int(s['primary_unit_id']) if pd.notna(s['primary_unit_id']) else 0,
                    'room_id': None, 'bed_id': None, 'encounter_id': None, 'shift_id': shift_id,
                    'assignment_start_datetime': shift_start, 'assignment_end_datetime': None,
                    'assignment_role': s['role_name'], 'patient_load': 0, 'status': 'Active', 'source_run_id': run_id,
                    'created_datetime': pd.Timestamp.utcnow().tz_localize(None), 'updated_datetime': pd.Timestamp.utcnow().tz_localize(None),
                })
                next_id += 1
    return query_db('SELECT * FROM dbo.ops_staff_assignment', engine)


def compute_staffing_state(engine=None, run_id=None, staffing_absence_rate=0.0):
    engine = engine or SINGLE_ENGINE
    units = query_db('SELECT unit_id, nurse_ratio_target FROM dbo.ops_unit', engine)
    if units.empty:
        return pd.DataFrame()
    now = pd.Timestamp.utcnow().tz_localize(None)
    rng = random.Random()
    with transaction(engine) as conn:
        for _, u in units.iterrows():
            unit_id = int(u['unit_id'])
            occupied = query_db(
                "SELECT COUNT(*) AS n FROM dbo.ops_bed_state bs JOIN dbo.beds b ON bs.bed_id = b.bed_id "
                "JOIN dbo.ops_unit uu ON uu.hospital_id = b.hospital_id AND uu.source_floor_number = b.floor_number "
                f"WHERE uu.unit_id = {unit_id} AND bs.occupancy_status = 'Occupied'", conn)
            occupied_beds = int(occupied.iloc[0]['n']) if not occupied.empty else 0
            required_rn = max(1, round(occupied_beds / max(u['nurse_ratio_target'], 0.5)))
            scheduled = query_db(
                "SELECT s.role_code, COUNT(*) AS n FROM dbo.ops_staff_assignment sa JOIN dbo.ops_staff s ON sa.staff_id = s.staff_id "
                f"WHERE sa.unit_id = {unit_id} AND sa.status = 'Active' GROUP BY s.role_code", conn)
            scheduled_rn = int(scheduled.loc[scheduled['role_code'].isin(['RN', 'CN']), 'n'].sum()) if not scheduled.empty else 0
            scheduled_pct = int(scheduled.loc[scheduled['role_code'] == 'PCT', 'n'].sum()) if not scheduled.empty else 0
            present_rn = scheduled_rn
            if staffing_absence_rate > 0:
                absences = sum(1 for _ in range(scheduled_rn) if rng.random() < staffing_absence_rate)
                present_rn = max(0, scheduled_rn - absences)
            present_pct = scheduled_pct
            charge = query_db(
                "SELECT TOP 1 sa.staff_id FROM dbo.ops_staff_assignment sa JOIN dbo.ops_staff s ON sa.staff_id = s.staff_id "
                f"WHERE sa.unit_id = {unit_id} AND sa.status = 'Active' AND s.role_code = 'CN'", conn)
            charge_id = int(charge.iloc[0]['staff_id']) if not charge.empty else None
            open_shift = max(0, required_rn - present_rn)
            coverage_pct = round(100.0 * present_rn / required_rn, 1) if required_rn > 0 else 100.0
            status = 'Critical' if coverage_pct < 70 else ('Watch' if coverage_pct < 90 else 'Normal')
            upsert_row(conn, 'ops_staffing_state', ['unit_id'], {
                'unit_id': unit_id, 'snapshot_datetime': now, 'scheduled_rn_count': scheduled_rn,
                'present_rn_count': present_rn, 'required_rn_count': required_rn, 'scheduled_pct_count': scheduled_pct,
                'present_pct_count': present_pct, 'open_shift_count': open_shift, 'charge_nurse_staff_id': charge_id,
                'staffing_coverage_pct': coverage_pct, 'staffing_status': status, 'simulation_run_id': run_id,
            })
    return query_db('SELECT * FROM dbo.ops_staffing_state', engine)


In [ ]:
# ============================================================================
# CELL: Equipment Inventory Generation + Initial Equipment State
# ============================================================================
_MANUFACTURERS = ['Contoso Med', 'Fabrikam Health', 'Northwind Devices', 'AdventureWorks Medical']


def _resolve_count(count_expr, licensed_beds):
    if isinstance(count_expr, tuple) and count_expr[0] == 'per_bed':
        return max(1, int(-(-licensed_beds * count_expr[1] // 1)))
    return int(count_expr)


def generate_equipment(engine=None, seed=None):
    engine = engine or SINGLE_ENGINE
    units = query_db('SELECT unit_id, hospital_id, building_id, source_floor_number, unit_type, licensed_bed_count FROM dbo.ops_unit', engine)
    if units.empty:
        raise RuntimeError('No ops_unit rows found -- run generate_hierarchy() first.')
    existing = query_db('SELECT DISTINCT unit_id FROM dbo.ops_equipment', engine)
    equipped_units = set(existing['unit_id'].tolist()) if not existing.empty else set()
    fake = Faker()
    if seed is not None:
        Faker.seed(seed)
        random.seed(seed)
    rng = random.Random(seed)

    to_equip = units[~units['unit_id'].isin(equipped_units)]
    if not to_equip.empty:
        next_id = get_next_id('ops_equipment', 'equipment_id', engine)
        with transaction(engine) as conn:
            for _, u in to_equip.iterrows():
                unit_type = u['unit_type'] if pd.notna(u['unit_type']) else 'default'
                plan = EQUIPMENT_BY_UNIT_TYPE.get(unit_type, EQUIPMENT_BY_UNIT_TYPE['default'])
                for equipment_type, count_expr in plan:
                    count = _resolve_count(count_expr, int(u['licensed_bed_count']) or 1)
                    for _ in range(count):
                        criticality = 'Critical' if equipment_type in CRITICAL_EQUIPMENT_TYPES else 'Standard'
                        upsert_row(conn, 'ops_equipment', ['equipment_id'], {
                            'equipment_id': next_id, 'hospital_id': int(u['hospital_id']), 'building_id': int(u['building_id']),
                            'floor_number': int(u['source_floor_number']), 'unit_id': int(u['unit_id']), 'room_id': None,
                            'equipment_type': equipment_type, 'manufacturer': rng.choice(_MANUFACTURERS),
                            'model': f"{equipment_type.split()[0][:3].upper()}-{rng.randint(100, 999)}",
                            'synthetic_serial_number': f'SN-{next_id:07d}', 'criticality': criticality,
                            'installation_date': (pd.Timestamp.utcnow() - pd.Timedelta(days=rng.randint(30, 2000))).date(),
                            'maintenance_interval_days': 90 if criticality == 'Critical' else 180, 'active_flag': True,
                            'created_datetime': pd.Timestamp.utcnow().tz_localize(None), 'updated_datetime': pd.Timestamp.utcnow().tz_localize(None),
                        })
                        next_id += 1
    return query_db('SELECT * FROM dbo.ops_equipment', engine)


def initialize_equipment_state(engine=None, run_id=None):
    engine = engine or SINGLE_ENGINE
    equipment = query_db('SELECT equipment_id, maintenance_interval_days FROM dbo.ops_equipment', engine)
    existing = query_db('SELECT equipment_id FROM dbo.ops_equipment_state', engine)
    existing_ids = set(existing['equipment_id'].tolist()) if not existing.empty else set()
    to_init = equipment[~equipment['equipment_id'].isin(existing_ids)]
    now = pd.Timestamp.utcnow().tz_localize(None)
    if not to_init.empty:
        with transaction(engine) as conn:
            for _, eq in to_init.iterrows():
                upsert_row(conn, 'ops_equipment_state', ['equipment_id'], {
                    'equipment_id': int(eq['equipment_id']), 'status': 'Available', 'availability_status': 'Available',
                    'utilization_status': 'Idle', 'battery_pct': 100.0, 'temperature': None, 'pressure': None, 'error_code': None,
                    'maintenance_due_date': (now + pd.Timedelta(days=int(eq['maintenance_interval_days']))).date(),
                    'last_seen_datetime': now, 'updated_datetime': now, 'simulation_run_id': run_id,
                })
    return query_db('SELECT * FROM dbo.ops_equipment_state', engine)


In [ ]:
# ============================================================================
# CELL: Room/Bed Operational-State Initialization + Iteration Helpers
# Current-state tables (ops_room_state, ops_bed_state) are upserted, never
# truncated. Initial state is derived from `patient_bed_assignments`.
# ============================================================================

def initialize_bed_state(engine=None, run_id=None):
    engine = engine or SINGLE_ENGINE
    beds = query_db('SELECT bed_id, hospital_id FROM dbo.beds', engine)
    existing = query_db('SELECT bed_id FROM dbo.ops_bed_state', engine)
    existing_ids = set(existing['bed_id'].tolist()) if not existing.empty else set()
    open_assignments = query_db(
        'SELECT bed_id, admission_id, patient_id, assigned_datetime FROM dbo.patient_bed_assignments WHERE discharged_datetime IS NULL', engine)
    open_by_bed = {int(r['bed_id']): r for _, r in open_assignments.iterrows()} if not open_assignments.empty else {}
    admissions = query_db('SELECT admission_id, encounter_id FROM dbo.admissions', engine)
    encounter_by_admission = dict(zip(admissions['admission_id'], admissions['encounter_id'])) if not admissions.empty else {}

    to_init = beds[~beds['bed_id'].isin(existing_ids)]
    now = pd.Timestamp.utcnow().tz_localize(None)
    if not to_init.empty:
        with transaction(engine) as conn:
            for _, b in to_init.iterrows():
                bed_id = int(b['bed_id'])
                occ = open_by_bed.get(bed_id)
                if occ is not None:
                    admission_id = int(occ['admission_id']) if pd.notna(occ['admission_id']) else None
                    upsert_row(conn, 'ops_bed_state', ['bed_id'], {
                        'bed_id': bed_id, 'operational_status': 'Occupied', 'occupancy_status': 'Occupied',
                        'encounter_id': encounter_by_admission.get(admission_id), 'patient_id': int(occ['patient_id']),
                        'admission_id': admission_id, 'assigned_datetime': occ['assigned_datetime'],
                        'expected_release_datetime': None, 'cleaning_required_flag': False,
                        'updated_datetime': now, 'simulation_run_id': run_id,
                    })
                else:
                    upsert_row(conn, 'ops_bed_state', ['bed_id'], {
                        'bed_id': bed_id, 'operational_status': 'Available', 'occupancy_status': 'Available',
                        'encounter_id': None, 'patient_id': None, 'admission_id': None, 'assigned_datetime': None,
                        'expected_release_datetime': None, 'cleaning_required_flag': False,
                        'updated_datetime': now, 'simulation_run_id': run_id,
                    })
    return query_db('SELECT * FROM dbo.ops_bed_state', engine)


def initialize_room_state(engine=None, run_id=None):
    engine = engine or SINGLE_ENGINE
    rooms = query_db('SELECT room_id, bed_count FROM dbo.rooms', engine)
    existing = query_db('SELECT room_id FROM dbo.ops_room_state', engine)
    existing_ids = set(existing['room_id'].tolist()) if not existing.empty else set()
    to_init = rooms[~rooms['room_id'].isin(existing_ids)]
    now = pd.Timestamp.utcnow().tz_localize(None)
    if not to_init.empty:
        with transaction(engine) as conn:
            for _, r in to_init.iterrows():
                room_id = int(r['room_id'])
                occ = query_db(f"SELECT COUNT(*) AS n FROM dbo.ops_bed_state bs JOIN dbo.beds b ON bs.bed_id = b.bed_id "
                               f"WHERE b.room_id = {room_id} AND bs.occupancy_status = 'Occupied'", conn)
                occupied = int(occ.iloc[0]['n']) if not occ.empty else 0
                available = max(0, int(r['bed_count']) - occupied)
                upsert_row(conn, 'ops_room_state', ['room_id'], {
                    'room_id': room_id, 'operational_status': 'Occupied' if occupied > 0 else 'Available',
                    'isolation_status': 'None', 'cleaning_status': 'Clean', 'maintenance_status': 'None',
                    'current_patient_count': occupied, 'available_bed_count': available,
                    'last_cleaned_datetime': now, 'next_expected_available_datetime': None,
                    'updated_datetime': now, 'simulation_run_id': run_id,
                })
    return query_db('SELECT * FROM dbo.ops_room_state', engine)


def refresh_room_aggregates(engine=None, run_id=None):
    engine = engine or SINGLE_ENGINE
    rooms = query_db('SELECT room_id, bed_count FROM dbo.rooms', engine)
    now = pd.Timestamp.utcnow().tz_localize(None)
    with transaction(engine) as conn:
        for _, r in rooms.iterrows():
            room_id = int(r['room_id'])
            occ = query_db(f"SELECT COUNT(*) AS n FROM dbo.ops_bed_state bs JOIN dbo.beds b ON bs.bed_id = b.bed_id "
                           f"WHERE b.room_id = {room_id} AND bs.occupancy_status = 'Occupied'", conn)
            occupied = int(occ.iloc[0]['n']) if not occ.empty else 0
            available = max(0, int(r['bed_count']) - occupied)
            upsert_row(conn, 'ops_room_state', ['room_id'], {
                'room_id': room_id, 'operational_status': 'Occupied' if occupied > 0 else 'Available',
                'isolation_status': 'None', 'cleaning_status': 'Clean', 'maintenance_status': 'None',
                'current_patient_count': occupied, 'available_bed_count': available,
                'last_cleaned_datetime': now, 'next_expected_available_datetime': None,
                'updated_datetime': now, 'simulation_run_id': run_id,
            })


def simulate_room_cleaning_iteration(engine=None, run_id=None, rng=None):
    engine = engine or SINGLE_ENGINE
    rng = rng or random.Random()
    cleaning_rooms = query_db(
        "SELECT DISTINCT r.room_id FROM dbo.rooms r JOIN dbo.beds b ON r.room_id = b.room_id "
        "JOIN dbo.ops_bed_state bs ON b.bed_id = bs.bed_id WHERE bs.occupancy_status = 'Cleaning'", engine)
    changed = 0
    if not cleaning_rooms.empty:
        now = pd.Timestamp.utcnow().tz_localize(None)
        with transaction(engine) as conn:
            for _, r in cleaning_rooms.iterrows():
                if rng.random() < 0.4:
                    conn.execute(text(
                        "UPDATE dbo.ops_bed_state SET occupancy_status = 'Available', operational_status = 'Available', "
                        "cleaning_required_flag = 0, updated_datetime = :now, simulation_run_id = :rid "
                        "WHERE bed_id IN (SELECT bed_id FROM dbo.beds WHERE room_id = :room_id) AND occupancy_status = 'Cleaning'"
                    ), {'now': now, 'rid': run_id, 'room_id': int(r['room_id'])})
                    changed += 1
    if changed:
        refresh_room_aggregates(engine, run_id)
    return changed


In [ ]:
from __future__ import annotations
from typing import Optional
from sqlalchemy import text
from sqlalchemy.engine import Engine
_MIN_DWELL_HOURS = 2.0

# Hard ceiling on a single drawn stay, as a multiple of that category's mean.
# Real length-of-stay distributions have a long right tail, but an unbounded
# lognormal draw can occasionally produce a year-long demo stay.
_LOS_OUTLIER_CAP_MULTIPLE = 6.0


def _draw_los_hours(mean_days: float, median_days: float, rng: random.Random) -> float:
    """Draw one length of stay (hours) from a right-skewed lognormal fitted to
    a published mean and median.

    Real length-of-stay distributions are right-skewed: a few long stays pull
    the mean above the median. A lognormal is fully determined by those two
    published statistics, so the fit is derived rather than guessed:

        median = exp(mu)              =>  mu    = ln(median)
        mean   = exp(mu + sigma^2/2)  =>  sigma = sqrt(2 * ln(mean / median))

    Falls back to a symmetric jitter when a source reports no usable median or
    a median at/above the mean (which no lognormal can represent).
    """
    mean_days = max(float(mean_days), 0.01)
    median_days = max(float(median_days), 0.0)
    if median_days <= 0 or median_days >= mean_days:
        drawn = mean_days * rng.uniform(0.7, 1.3)
    else:
        mu = math.log(median_days)
        sigma = math.sqrt(2.0 * math.log(mean_days / median_days))
        drawn = rng.lognormvariate(mu, sigma)
    capped = min(drawn, mean_days * _LOS_OUTLIER_CAP_MULTIPLE)
    return max(capped * 24.0, _MIN_DWELL_HOURS)


# Published length-of-stay benchmarks keyed by the same clinical_family values
# used by the icd_reference dimension, so a simulated stay is driven by what
# the patient has rather than by how fast the simulation clock is running.
# Sources, derivations and confidence grades: docs/LENGTH_OF_STAY_BENCHMARKS.md
#   ed_dwell_hours  - median ED dwell for admitted patients
#   los_mean_days   - mean inpatient length of stay
#   los_median_days - median inpatient length of stay (fits the lognormal)
#   icu_share       - published share of *this condition's* admissions needing
#                     critical care (a cohort rate, calibrated below)
#   icu_los_days    - mean critical-care length of stay
_LOS_BENCHMARKS = {
    "Cardiology":                 {"ed_dwell_hours": 4.0, "los_mean_days": 4.5, "los_median_days": 3.0, "icu_share": 0.55, "icu_los_days": 2.5},
    "Respiratory":                {"ed_dwell_hours": 4.0, "los_mean_days": 4.5, "los_median_days": 3.0, "icu_share": 0.45, "icu_los_days": 3.5},
    "Endocrine & Metabolic":      {"ed_dwell_hours": 4.5, "los_mean_days": 5.5, "los_median_days": 4.0, "icu_share": 0.44, "icu_los_days": 2.0},
    "Injury & Trauma":            {"ed_dwell_hours": 4.0, "los_mean_days": 6.0, "los_median_days": 4.0, "icu_share": 0.25, "icu_los_days": 3.5},
    "Musculoskeletal":            {"ed_dwell_hours": 5.0, "los_mean_days": 4.0, "los_median_days": 3.0, "icu_share": 0.10, "icu_los_days": 2.0},
    "Behavioral Health":          {"ed_dwell_hours": 10.0, "los_mean_days": 7.6, "los_median_days": 6.0, "icu_share": 0.02, "icu_los_days": 2.0},
    "Gastroenterology":           {"ed_dwell_hours": 4.0, "los_mean_days": 4.0, "los_median_days": 3.0, "icu_share": 0.43, "icu_los_days": 2.5},
    "Oncology":                   {"ed_dwell_hours": 5.5, "los_mean_days": 6.5, "los_median_days": 5.0, "icu_share": 0.12, "icu_los_days": 3.0},
    "Rehabilitation & Aftercare": {"ed_dwell_hours": 3.0, "los_mean_days": 8.0, "los_median_days": 6.0, "icu_share": 0.02, "icu_los_days": 2.0},
    "Signs & Symptoms":           {"ed_dwell_hours": 3.5, "los_mean_days": 2.5, "los_median_days": 1.5, "icu_share": 0.15, "icu_los_days": 1.5},
    "Neurology":                  {"ed_dwell_hours": 3.0, "los_mean_days": 5.5, "los_median_days": 4.0, "icu_share": 0.45, "icu_los_days": 3.0},
    "Infectious Disease":         {"ed_dwell_hours": 4.0, "los_mean_days": 6.0, "los_median_days": 4.0, "icu_share": 0.40, "icu_los_days": 4.0},
    "Ear, Nose & Throat":         {"ed_dwell_hours": 3.0, "los_mean_days": 2.5, "los_median_days": 2.0, "icu_share": 0.05, "icu_los_days": 1.5},
    "Hematology":                 {"ed_dwell_hours": 4.0, "los_mean_days": 5.0, "los_median_days": 3.5, "icu_share": 0.15, "icu_los_days": 2.5},
    "Preventive & Aftercare":     {"ed_dwell_hours": 2.5, "los_mean_days": 2.0, "los_median_days": 1.5, "icu_share": 0.02, "icu_los_days": 1.5},
    "Renal & Genitourinary":      {"ed_dwell_hours": 4.5, "los_mean_days": 4.0, "los_median_days": 3.0, "icu_share": 0.48, "icu_los_days": 2.5},
}

# All-causes fallback row (docs/LENGTH_OF_STAY_BENCHMARKS.md section 5).
_BASELINE_LOS = {"ed_dwell_hours": 4.0, "los_mean_days": 5.1, "los_median_days": 3.0,
                 "icu_share": 0.18, "icu_los_days": 3.9}

# Published share of ED admissions that go to critical care (NHAMCS, with
# "unknown" redistributed proportionally). The per-family `icu_share` values
# above are *within-condition* cohort rates -- 70% of AMI admissions reach a
# CCU -- so using them directly would put most of the census in the ICU. They
# are rescaled by a single factor to preserve the published ordering between
# conditions while landing the blended rate on the published aggregate.
_ICU_TARGET_SHARE = 0.18
_ICU_CALIBRATION = _ICU_TARGET_SHARE / (
    sum(row["icu_share"] for row in _LOS_BENCHMARKS.values()) / len(_LOS_BENCHMARKS)
)


def _los_profile(clinical_family: Optional[str]) -> dict:
    """Benchmark row for a clinical family, falling back to all-causes."""
    return _LOS_BENCHMARKS.get(clinical_family or "", _BASELINE_LOS)


def _icu_admit_probability(clinical_family: Optional[str]) -> float:
    """Calibrated probability that this family's stay needs critical care."""
    raw = _los_profile(clinical_family)["icu_share"]
    return min(1.0, max(0.0, raw * _ICU_CALIBRATION))


# Each story is (icd_code, description, chief_complaint, drug, dosage,
# frequency, lab test, value, units, reference_range, clinical_family).
# The clinical_family must match a key in icd_reference.clinical_family so the
# stay inherits that family's published length-of-stay benchmarks.
_CLINICAL_STORIES = {
    "general": ("J18.9", "Pneumonia, unspecified organism", "Shortness of breath", "Ceftriaxone", "1 g", "Once daily", "White blood cell count", "14.2", "10^3/uL", "4.0-11.0", "Respiratory"),
    "pediatric": ("J45.901", "Unspecified asthma with (acute) exacerbation", "Wheezing", "Albuterol", "2.5 mg", "As needed", "Arterial oxygen partial pressure", "68", "mmHg", "80-100", "Respiratory"),
    "cardiac": ("I50.9", "Heart failure, unspecified", "Shortness of breath", "Furosemide", "40 mg", "Once daily", "BNP", "350", "pg/mL", "0-100", "Cardiology"),
    "cancer": ("D70.9", "Neutropenia, unspecified", "Fever", "Cefepime", "2 g", "Twice daily", "White blood cell count", "1.2", "10^3/uL", "4.0-11.0", "Hematology"),
    "rehab": ("M62.81", "Muscle weakness (generalized)", "Weakness", "Acetaminophen", "500 mg", "As needed", "Creatinine", "1.0", "mg/dL", "0.6-1.3", "Musculoskeletal"),
}


def _flow_query(sql, engine, params=None):
    return pd.read_sql(text(sql), engine, params=params or {})


def _flow_next_id(table, key, engine):
    value = _flow_query(f"SELECT MAX({key}) AS max_id FROM dbo.{table}", engine).iloc[0]["max_id"]
    return 1 if pd.isna(value) else int(value) + 1


def _insert_clinical_story(conn, encounter_id: int, patient_id: int, hospital_specialty: str,
                           provider_id: int, simulated_now: pd.Timestamp) -> str:
    """Create one traceable synthetic diagnosis, order and result for this stay.

    Returns the clinical family of the diagnosis so the caller can drive length
    of stay from that family's published benchmarks.
    """
    code, description, complaint, drug, dosage, frequency, test, value, units, reference, family = _CLINICAL_STORIES.get(
        hospital_specialty, _CLINICAL_STORIES["general"]
    )
    when = simulated_now.date()
    updated = conn.execute(text(
        "UPDATE dbo.encounters SET chief_complaint = :complaint "
        "WHERE encounter_id = :encounter_id AND patient_id = :patient_id"
    ), {"complaint": complaint, "encounter_id": encounter_id, "patient_id": patient_id})
    if updated.rowcount != 1:
        raise RuntimeError(f"Encounter {encounter_id} cannot be linked to patient {patient_id}")
    for table, key, columns, values in (
        ("diagnoses", "diagnosis_id",
         "encounter_id, patient_id, icd_code, description, onset_date, status",
         {"encounter_id": encounter_id, "patient_id": patient_id, "icd_code": code,
          "description": description, "onset_date": when, "status": "Active"}),
        ("medications", "medication_id",
         "encounter_id, patient_id, drug_name, dosage, frequency, start_date, end_date, prescribing_provider_id, status",
         {"encounter_id": encounter_id, "patient_id": patient_id, "drug_name": drug,
          "dosage": dosage, "frequency": frequency, "start_date": when,
          "end_date": None, "prescribing_provider_id": provider_id, "status": "Active"}),
        ("labs", "lab_id",
         "encounter_id, patient_id, test_name, test_code, order_date, result_date, value, units, reference_range, status",
         {"encounter_id": encounter_id, "patient_id": patient_id, "test_name": test,
          "test_code": None, "order_date": when, "result_date": when,
          "value": value,
          "units": units, "reference_range": reference, "status": "Completed"}),
    ):
        # IDs in the ODS are allocated by the daily generator using MAX(id)+1.
        # Hold the table lock until this transaction commits to avoid colliding
        # with a concurrent catch-up or a second simulator.
        row = conn.execute(text(
            f"SELECT COALESCE(MAX({key}), 0) + 1 FROM dbo.{table} WITH (TABLOCKX, HOLDLOCK)"
        )).scalar_one()
        params = {"new_id": int(row), **values}
        conn.execute(text(
            f"INSERT INTO dbo.{table} ({key}, {columns}) VALUES "
            f"(:new_id, {', '.join(':' + name for name in values)})"
        ), params)
    return family


def _pick_target_unit(units: pd.DataFrame, hospital_id: int, encounter_type: str, rng: random.Random) -> Optional[pd.Series]:
    hosp_units = units[units["hospital_id"] == hospital_id]
    if hosp_units.empty:
        return None
    if encounter_type == "Emergency":
        preferred = hosp_units[hosp_units["unit_type"] == "Emergency"]
        if preferred.empty:
            return None
        return preferred.sample(n=1, random_state=rng.randint(0, 2**31)).iloc[0]
    pool = hosp_units[hosp_units["unit_type"] != "Emergency"]
    if pool.empty:
        return None
    non_critical = pool[pool["unit_type"] != "Critical Care"]
    pool = non_critical if not non_critical.empty else pool
    return pool.sample(n=1, random_state=rng.randint(0, 2**31)).iloc[0]


def _find_available_bed(engine: Engine, hospital_id: int, floor_number: int) -> Optional[int]:
    df = _flow_query(
        "SELECT TOP 1 bs.bed_id FROM dbo.ops_bed_state bs JOIN dbo.beds b ON bs.bed_id = b.bed_id "
        "WHERE b.hospital_id = :h AND b.floor_number = :f AND bs.occupancy_status = 'Available' ORDER BY bs.bed_id",
        engine, {"h": hospital_id, "f": floor_number},
    )
    return int(df.iloc[0]["bed_id"]) if not df.empty else None


def admit_patients(engine: Engine, simulated_now: pd.Timestamp, run_id: int, batch_size: int,
                    scenario: ScenarioProfile, rng: random.Random) -> int:
    """Assign an operational bed to up to `batch_size` active admissions that
    don't yet have one. Returns count of admissions processed."""
    candidates = _flow_query(
        "SELECT TOP (:lim) a.admission_id, a.encounter_id, a.patient_id, a.hospital_id, "
        "e.encounter_type, e.provider_id, h.specialty "
        "FROM dbo.admissions a JOIN dbo.encounters e ON a.encounter_id = e.encounter_id "
        "JOIN dbo.hospitals h ON h.hospital_id = a.hospital_id "
        "WHERE a.discharge_datetime IS NULL AND a.patient_id = e.patient_id "
        "AND a.hospital_id = e.hospital_id AND a.admit_datetime <= :now "
        "AND a.admit_datetime >= DATEADD(DAY, -1, :now) "
        "AND NOT EXISTS (SELECT 1 FROM dbo.ops_patient_movement pm "
        "WHERE pm.encounter_id = a.encounter_id AND pm.movement_type = 'Admission') "
        "AND NOT EXISTS (SELECT 1 FROM dbo.ops_simulated_episode se WHERE se.encounter_id = a.encounter_id) "
        "AND NOT EXISTS (SELECT 1 FROM dbo.ops_bed_state bs WHERE bs.encounter_id = a.encounter_id) "
        "ORDER BY a.admit_datetime DESC",
        engine, {"lim": batch_size * 20, "now": simulated_now},
    )
    if candidates.empty:
        return 0
    units = _flow_query("SELECT unit_id, hospital_id, source_floor_number, unit_type FROM dbo.ops_unit", engine)
    processed = 0
    bed_event_id = _flow_next_id("ops_bed_state_event", "bed_state_event_id", engine)
    movement_id = _flow_next_id("ops_patient_movement", "movement_id", engine)
    with transaction(engine) as conn:
        for _, cand in candidates.iterrows():
            if processed >= batch_size:
                break
            unit = _pick_target_unit(units, int(cand["hospital_id"]), cand["encounter_type"], rng)
            if unit is None:
                continue
            bed_id = _find_available_bed(conn, int(cand["hospital_id"]), int(unit["source_floor_number"]))
            if bed_id is None:
                continue  # no capacity -- alert_engine will surface this as a capacity alert
            family = _insert_clinical_story(conn, int(cand["encounter_id"]), int(cand["patient_id"]),
                                            str(cand["specialty"]), int(cand["provider_id"]), simulated_now)
            profile = _los_profile(family)
            # Draw this stay's length from its family's published benchmarks.
            # Scenario delay is applied to the drawn stay rather than to a
            # per-iteration probability, so length of stay stays a function of
            # the condition and not of SPEED_MULTIPLIER.
            target_los_hours = _draw_los_hours(
                profile["los_mean_days"], profile["los_median_days"], rng
            ) * max(0.1, scenario.discharge_delay_multiplier)
            ed_dwell_hours = max(0.25, profile["ed_dwell_hours"] * rng.uniform(0.6, 1.6))
            icu_expected = rng.random() < _icu_admit_probability(family)
            if icu_expected:
                target_los_hours += profile["icu_los_days"] * 24.0
            # Round before deriving the timestamp so the stored target and the
            # expected discharge always reconcile for anyone comparing
            # target_los_hours against DATEDIFF on the episode.
            target_los_hours = round(float(target_los_hours), 2)
            expected_discharge = simulated_now + pd.Timedelta(hours=target_los_hours)
            insert_rows(conn, "ops_simulated_episode", [{
                "encounter_id": int(cand["encounter_id"]), "admission_id": int(cand["admission_id"]),
                "patient_id": int(cand["patient_id"]), "simulation_run_id": run_id,
                "created_datetime": simulated_now,
                "icd_family": family,
                "target_los_hours": target_los_hours,
                "ed_dwell_hours": round(float(ed_dwell_hours), 2),
                "icu_expected_flag": bool(icu_expected),
                "expected_discharge_datetime": expected_discharge,
            }])
            upsert_row(conn, "ops_bed_state", ["bed_id"], {
                "bed_id": bed_id, "operational_status": "Occupied", "occupancy_status": "Occupied",
                "encounter_id": int(cand["encounter_id"]), "patient_id": int(cand["patient_id"]),
                "admission_id": int(cand["admission_id"]), "assigned_datetime": simulated_now,
                "expected_release_datetime": expected_discharge, "cleaning_required_flag": False,
                "updated_datetime": simulated_now, "simulation_run_id": run_id,
            })
            insert_rows(conn, "ops_bed_state_event", [{
                "bed_state_event_id": bed_event_id, "bed_id": bed_id, "event_datetime": simulated_now,
                "event_type": "Admission", "status_before": "Available", "status_after": "Occupied",
                "encounter_id": int(cand["encounter_id"]), "patient_id": int(cand["patient_id"]),
                "reason": "Simulated admission bed assignment", "simulation_run_id": run_id,
            }])
            insert_rows(conn, "ops_patient_movement", [{
                "movement_id": movement_id, "encounter_id": int(cand["encounter_id"]), "patient_id": int(cand["patient_id"]),
                "hospital_id": int(cand["hospital_id"]), "from_unit_id": None, "from_room_id": None, "from_bed_id": None,
                "to_unit_id": int(unit["unit_id"]), "to_room_id": None, "to_bed_id": bed_id,
                "requested_datetime": simulated_now, "accepted_datetime": simulated_now,
                "started_datetime": simulated_now, "completed_datetime": simulated_now,
                "movement_type": "Admission", "movement_status": "Completed", "priority": "Routine",
                "delay_reason": None, "simulation_run_id": run_id,
            }])
            upsert_row(conn, "ops_discharge_readiness", ["encounter_id"], {
                "encounter_id": int(cand["encounter_id"]), "patient_id": int(cand["patient_id"]),
                "expected_discharge_datetime": expected_discharge, "readiness_status": "Not Ready",
                "clinical_ready_flag": False, "medication_ready_flag": False, "transport_ready_flag": False,
                "destination_ready_flag": False, "education_complete_flag": False,
                "outstanding_barrier_count": 1, "primary_barrier": "Simulated: awaiting clinical progress",
                "updated_datetime": simulated_now, "simulation_run_id": run_id,
            })
            bed_event_id += 1
            movement_id += 1
            processed += 1
    return processed


def _split_batch(batch_size: int) -> tuple[int, int]:
    """Reserve roughly half of a batch for simulator-created episodes.

    A live database carries a large backlog of pre-existing operational rows
    whose timestamps are always older than a freshly created episode. Selecting
    purely oldest-first therefore starves new stays, so they never progress past
    admission. Reserving part of each batch keeps new journeys moving while
    preserving ambient movement for the legacy population.
    """
    simulated_limit = max(1, batch_size // 2)
    return simulated_limit, max(0, batch_size - simulated_limit)


def advance_discharge_readiness(engine: Engine, simulated_now: pd.Timestamp, run_id: int,
                                 batch_size: int, discharge_delay_multiplier: float, rng: random.Random) -> int:
    """Probabilistically progress discharge-readiness flags for occupied beds
    (all values explicitly SIMULATED, never a real clinical judgement)."""
    simulated_limit, legacy_limit = _split_batch(batch_size)
    occupied = _flow_query(
        "SELECT encounter_id, outstanding_barrier_count, expected_release_datetime, is_simulated FROM ("
        "  SELECT dr.encounter_id, dr.outstanding_barrier_count, bs.expected_release_datetime,"
        "    CASE WHEN se.encounter_id IS NULL THEN 0 ELSE 1 END AS is_simulated,"
        "    ROW_NUMBER() OVER (PARTITION BY CASE WHEN se.encounter_id IS NULL THEN 0 ELSE 1 END"
        "      ORDER BY dr.updated_datetime ASC) AS rn"
        "  FROM dbo.ops_discharge_readiness dr"
        "  LEFT JOIN dbo.ops_simulated_episode se ON se.encounter_id = dr.encounter_id"
        "  LEFT JOIN dbo.ops_bed_state bs ON bs.encounter_id = dr.encounter_id"
        "  WHERE dr.readiness_status <> 'Ready'"
        ") ranked WHERE (is_simulated = 1 AND rn <= :sim_lim) OR (is_simulated = 0 AND rn <= :legacy_lim)",
        engine, {"sim_lim": simulated_limit, "legacy_lim": legacy_limit},
    )
    if occupied.empty:
        return 0
    progressed = 0
    # Legacy rows carry no drawn length of stay, so they keep the original
    # probabilistic walk to preserve ambient movement in the demo database.
    progress_chance = max(0.02, 0.20 / max(discharge_delay_multiplier, 0.1))
    with transaction(engine) as conn:
        for _, row in occupied.iterrows():
            expected_release = row["expected_release_datetime"]
            if int(row["is_simulated"]) == 1 and pd.notna(expected_release):
                # A simulated stay clears its barriers only once the simulated
                # clock reaches the length of stay drawn for its condition.
                if simulated_now < pd.Timestamp(expected_release):
                    continue
            elif rng.random() > progress_chance:
                continue
            barriers = max(0, int(row["outstanding_barrier_count"]) - 1)
            status = "Ready" if barriers == 0 else ("Pending" if barriers <= 1 else "Not Ready")
            conn.execute(text(
                "UPDATE dbo.ops_discharge_readiness SET outstanding_barrier_count = :b, readiness_status = :s, "
                "clinical_ready_flag = CASE WHEN :b = 0 THEN 1 ELSE clinical_ready_flag END, "
                "primary_barrier = CASE WHEN :b = 0 THEN NULL ELSE primary_barrier END, "
                "expected_discharge_datetime = COALESCE(expected_discharge_datetime, "
                "  CASE WHEN :b = 0 THEN :now ELSE NULL END), "
                "updated_datetime = :now, simulation_run_id = :rid WHERE encounter_id = :eid"
            ), {"b": barriers, "s": status, "now": simulated_now, "rid": run_id, "eid": int(row["encounter_id"])})
            progressed += 1
    return progressed


def seed_discharge_readiness_for_occupied(engine: Engine, run_id: int | None = None) -> int:
    """Initialize readiness for pre-existing occupied beds during setup."""
    missing = _flow_query(
        "SELECT bs.encounter_id, bs.patient_id FROM dbo.ops_bed_state bs "
        "LEFT JOIN dbo.ops_discharge_readiness dr ON bs.encounter_id = dr.encounter_id "
        "WHERE bs.occupancy_status = 'Occupied' AND bs.encounter_id IS NOT NULL AND dr.encounter_id IS NULL", engine
    )
    if missing.empty:
        return 0
    now = pd.Timestamp.utcnow().tz_localize(None)
    with transaction(engine) as conn:
        for _, row in missing.iterrows():
            upsert_row(conn, "ops_discharge_readiness", ["encounter_id"], {
                "encounter_id": int(row["encounter_id"]), "patient_id": int(row["patient_id"]),
                "expected_discharge_datetime": None, "readiness_status": "Not Ready",
                "clinical_ready_flag": False, "medication_ready_flag": False, "transport_ready_flag": False,
                "destination_ready_flag": False, "education_complete_flag": False, "outstanding_barrier_count": 1,
                "primary_barrier": "Simulated: awaiting clinical progress", "updated_datetime": now,
                "simulation_run_id": run_id,
            })
    return len(missing)


def discharge_ready_patients(engine: Engine, simulated_now: pd.Timestamp, run_id: int, batch_size: int, rng: random.Random) -> int:
    """Release ready beds; complete clinical admissions for simulator-created stays."""
    simulated_limit, legacy_limit = _split_batch(batch_size)
    ready = _flow_query(
        "SELECT bed_id, encounter_id, patient_id, admission_id, assigned_datetime, hospital_id, unit_id, simulated_stay FROM ("
        "  SELECT bs.bed_id, bs.encounter_id, bs.patient_id, bs.admission_id, bs.assigned_datetime,"
        "    b.hospital_id, u.unit_id,"
        "    CASE WHEN se.encounter_id IS NULL THEN 0 ELSE 1 END AS simulated_stay,"
        "    ROW_NUMBER() OVER (PARTITION BY CASE WHEN se.encounter_id IS NULL THEN 0 ELSE 1 END"
        "      ORDER BY bs.assigned_datetime ASC) AS rn"
        "  FROM dbo.ops_bed_state bs"
        "  JOIN dbo.beds b ON bs.bed_id = b.bed_id"
        "  JOIN dbo.ops_unit u ON u.hospital_id = b.hospital_id AND u.source_floor_number = b.floor_number"
        "  LEFT JOIN dbo.ops_simulated_episode se ON se.encounter_id = bs.encounter_id"
        "    AND se.admission_id = bs.admission_id AND se.patient_id = bs.patient_id"
        "  JOIN dbo.ops_discharge_readiness dr ON bs.encounter_id = dr.encounter_id"
        "  WHERE bs.occupancy_status = 'Occupied' AND dr.readiness_status = 'Ready'"
        "    AND bs.assigned_datetime <= :cutoff"
        ") ranked WHERE (simulated_stay = 1 AND rn <= :sim_lim) OR (simulated_stay = 0 AND rn <= :legacy_lim)",
        engine,
        {"sim_lim": simulated_limit, "legacy_lim": legacy_limit,
         "cutoff": simulated_now - pd.Timedelta(hours=_MIN_DWELL_HOURS)},
    )
    if ready.empty:
        return 0
    bed_event_id = _flow_next_id("ops_bed_state_event", "bed_state_event_id", engine)
    movement_id = _flow_next_id("ops_patient_movement", "movement_id", engine)
    released = 0
    with transaction(engine) as conn:
        for _, row in ready.iterrows():
            bed_id = int(row["bed_id"])
            if int(row["simulated_stay"]) == 1:
                if pd.isna(row["admission_id"]):
                    raise RuntimeError(f"Simulated encounter {row['encounter_id']} lost its admission ID")
                result = conn.execute(text(
                    "UPDATE dbo.admissions SET discharge_datetime = :now "
                    "WHERE admission_id = :aid AND encounter_id = :eid AND patient_id = :pid "
                    "AND discharge_datetime IS NULL AND admit_datetime <= :now"
                ), {"now": simulated_now, "aid": int(row["admission_id"]),
                    "eid": int(row["encounter_id"]), "pid": int(row["patient_id"])})
                if result.rowcount != 1:
                    raise RuntimeError(f"Clinical discharge could not be recorded for admission {row['admission_id']}")
            upsert_row(conn, "ops_bed_state", ["bed_id"], {
                "bed_id": bed_id, "operational_status": "Cleaning", "occupancy_status": "Cleaning",
                "encounter_id": None, "patient_id": None, "admission_id": None,
                "assigned_datetime": None, "expected_release_datetime": None, "cleaning_required_flag": True,
                "updated_datetime": simulated_now, "simulation_run_id": run_id,
            })
            insert_rows(conn, "ops_bed_state_event", [{
                "bed_state_event_id": bed_event_id, "bed_id": bed_id, "event_datetime": simulated_now,
                "event_type": "Discharge", "status_before": "Occupied", "status_after": "Cleaning",
                "encounter_id": int(row["encounter_id"]), "patient_id": int(row["patient_id"]),
                "reason": "Simulated operational discharge (bed released for cleaning)", "simulation_run_id": run_id,
            }])
            insert_rows(conn, "ops_patient_movement", [{
                "movement_id": movement_id, "encounter_id": int(row["encounter_id"]), "patient_id": int(row["patient_id"]),
                "hospital_id": int(row["hospital_id"]), "from_unit_id": int(row["unit_id"]),
                "from_room_id": None, "from_bed_id": bed_id,
                "to_unit_id": None, "to_room_id": None, "to_bed_id": None,
                "requested_datetime": simulated_now, "accepted_datetime": simulated_now,
                "started_datetime": simulated_now, "completed_datetime": simulated_now,
                "movement_type": "Discharge", "movement_status": "Completed", "priority": "Routine",
                "delay_reason": None, "simulation_run_id": run_id,
            }])
            bed_event_id += 1
            movement_id += 1
            released += 1
    return released


def transfer_patients(engine: Engine, simulated_now: pd.Timestamp, run_id: int, batch_size: int,
                       icu_pressure_multiplier: float, rng: random.Random) -> int:
    """Occasionally move an occupied patient to a different unit on the same
    hospital (e.g. escalation to a Critical Care unit)."""
    simulated_limit, legacy_limit = _split_batch(batch_size)
    occupied = _flow_query(
        "SELECT bed_id, encounter_id, patient_id, admission_id, assigned_datetime, hospital_id, floor_number, "
        "expected_release_datetime, is_simulated, icu_expected_flag FROM ("
        "  SELECT bs.bed_id, bs.encounter_id, bs.patient_id, bs.admission_id, bs.assigned_datetime,"
        "    b.hospital_id, b.floor_number, bs.expected_release_datetime, se.icu_expected_flag,"
        "    CASE WHEN se.encounter_id IS NULL THEN 0 ELSE 1 END AS is_simulated,"
        "    ROW_NUMBER() OVER (PARTITION BY CASE WHEN se.encounter_id IS NULL THEN 0 ELSE 1 END"
        "      ORDER BY CASE WHEN u.unit_type = 'Emergency' THEN 0 ELSE 1 END, NEWID()) AS rn"
        "  FROM dbo.ops_bed_state bs JOIN dbo.beds b ON bs.bed_id = b.bed_id"
        "  JOIN dbo.ops_unit u ON u.hospital_id = b.hospital_id AND u.source_floor_number = b.floor_number"
        "  LEFT JOIN dbo.ops_simulated_episode se ON se.encounter_id = bs.encounter_id"
        "  WHERE bs.occupancy_status = 'Occupied' AND bs.assigned_datetime <= :cutoff"
        ") ranked WHERE (is_simulated = 1 AND rn <= :sim_lim) OR (is_simulated = 0 AND rn <= :legacy_lim)",
        engine,
        {"sim_lim": simulated_limit, "legacy_lim": legacy_limit,
         "cutoff": simulated_now - pd.Timedelta(hours=_MIN_DWELL_HOURS)},
    )
    if occupied.empty:
        return 0
    units = _flow_query("SELECT unit_id, hospital_id, source_floor_number, unit_type FROM dbo.ops_unit", engine)
    movement_id = _flow_next_id("ops_patient_movement", "movement_id", engine)
    bed_event_id = _flow_next_id("ops_bed_state_event", "bed_state_event_id", engine)
    moved = 0
    with transaction(engine) as conn:
        for _, row in occupied.iterrows():
            hosp_units = units[(units["hospital_id"] == row["hospital_id"]) &
                               (units["source_floor_number"] != row["floor_number"]) &
                               (units["unit_type"] != "Emergency")]
            if hosp_units.empty:
                continue
            source_units = units[(units["hospital_id"] == row["hospital_id"]) &
                                 (units["source_floor_number"] == row["floor_number"])]
            if source_units.empty:
                raise RuntimeError(f"No operational unit for occupied bed {row['bed_id']}")
            source_unit = source_units.iloc[0]
            from_emergency = source_unit["unit_type"] == "Emergency"
            if from_emergency:
                # ED patients move on quickly; the modelled decision is *where*
                # they go, not whether they leave. Only the share flagged at
                # admit escalates to critical care -- previously every ED
                # transfer was forced into Critical Care, which put roughly
                # twice the published proportion of the census in the ICU.
                chance = 0.60
                if int(row["is_simulated"]) == 1 and pd.notna(row["icu_expected_flag"]):
                    wants_icu = bool(row["icu_expected_flag"])
                else:
                    wants_icu = rng.random() < _ICU_TARGET_SHARE
            else:
                chance = min(0.5, 0.05 * icu_pressure_multiplier)
                wants_icu = False
            if rng.random() > chance:
                continue
            critical = hosp_units[hosp_units["unit_type"] == "Critical Care"]
            if wants_icu:
                if not critical.empty:
                    hosp_units = critical
            else:
                non_critical = hosp_units[hosp_units["unit_type"] != "Critical Care"]
                if not non_critical.empty:
                    hosp_units = non_critical
            target_unit = hosp_units.sample(n=1, random_state=rng.randint(0, 2**31)).iloc[0]
            new_bed_id = _find_available_bed(conn, int(row["hospital_id"]), int(target_unit["source_floor_number"]))
            if new_bed_id is None:
                continue
            old_bed_id = int(row["bed_id"])
            upsert_row(conn, "ops_bed_state", ["bed_id"], {
                "bed_id": new_bed_id, "operational_status": "Occupied", "occupancy_status": "Occupied",
                "encounter_id": int(row["encounter_id"]), "patient_id": int(row["patient_id"]),
                "admission_id": int(row["admission_id"]) if pd.notna(row["admission_id"]) else None,
                "assigned_datetime": simulated_now,
                "expected_release_datetime": row["expected_release_datetime"]
                    if pd.notna(row["expected_release_datetime"]) else None,
                "cleaning_required_flag": False,
                "updated_datetime": simulated_now, "simulation_run_id": run_id,
            })
            upsert_row(conn, "ops_bed_state", ["bed_id"], {
                "bed_id": old_bed_id, "operational_status": "Cleaning", "occupancy_status": "Cleaning",
                "encounter_id": None, "patient_id": None, "admission_id": None,
                "assigned_datetime": None, "expected_release_datetime": None, "cleaning_required_flag": True,
                "updated_datetime": simulated_now, "simulation_run_id": run_id,
            })
            insert_rows(conn, "ops_bed_state_event", [{
                "bed_state_event_id": bed_event_id, "bed_id": old_bed_id, "event_datetime": simulated_now,
                "event_type": "Internal Transfer", "status_before": "Occupied", "status_after": "Cleaning",
                "encounter_id": int(row["encounter_id"]), "patient_id": int(row["patient_id"]),
                "reason": "Simulated internal transfer", "simulation_run_id": run_id,
            }])
            insert_rows(conn, "ops_patient_movement", [{
                "movement_id": movement_id, "encounter_id": int(row["encounter_id"]), "patient_id": int(row["patient_id"]),
                "hospital_id": int(row["hospital_id"]), "from_unit_id": int(source_unit["unit_id"]),
                "from_room_id": None, "from_bed_id": old_bed_id,
                "to_unit_id": int(target_unit["unit_id"]), "to_room_id": None, "to_bed_id": new_bed_id,
                "requested_datetime": simulated_now, "accepted_datetime": simulated_now,
                "started_datetime": simulated_now, "completed_datetime": simulated_now,
                "movement_type": "Internal Transfer" if target_unit["unit_type"] != "Critical Care" else "ICU Transfer",
                "movement_status": "Completed", "priority": "Urgent" if target_unit["unit_type"] == "Critical Care" else "Routine",
                "delay_reason": None, "simulation_run_id": run_id,
            }])
            bed_event_id += 1
            movement_id += 1
            moved += 1
    return moved


def simulate_patient_flow_iteration(engine: Engine, simulated_now: pd.Timestamp, run_id: int,
                                     scenario: ScenarioProfile, batch_size: int, rng: random.Random) -> dict:
    """Run one bounded batch of patient-flow changes for this iteration."""
    admitted = admit_patients(engine, simulated_now, run_id, max(1, batch_size // 3), scenario, rng)
    progressed = advance_discharge_readiness(engine, simulated_now, run_id, batch_size, scenario.discharge_delay_multiplier, rng)
    transferred = transfer_patients(engine, simulated_now, run_id, max(1, batch_size // 4), scenario.icu_pressure_multiplier, rng)
    discharged = discharge_ready_patients(engine, simulated_now, run_id, max(1, batch_size // 3), rng)
    return {"admitted": admitted, "discharge_progressed": progressed, "discharged": discharged, "transferred": transferred}


In [ ]:
# ============================================================================
# CELL: Per-Iteration Equipment State Transitions
# ============================================================================
_EQUIPMENT_TRANSITIONS = {
    'Available': [('InUse', 0.15)],
    'InUse': [('Available', 0.20)],
    'Warning': [('Unavailable', 0.30), ('Available', 0.40)],
    'Unavailable': [('Maintenance', 0.50)],
    'Maintenance': [('Available', 0.60)],
}


def simulate_equipment_iteration(engine, simulated_now, run_id, equipment_failure_rate, batch_size, rng):
    states = query_db(
        "SELECT es.equipment_id, es.status, eq.equipment_type, eq.criticality FROM dbo.ops_equipment_state es "
        "JOIN dbo.ops_equipment eq ON es.equipment_id = eq.equipment_id ORDER BY NEWID() OFFSET 0 ROWS FETCH NEXT (:lim) ROWS ONLY",
        engine, {'lim': batch_size})
    if states.empty:
        return {'events': 0, 'failures': 0, 'recoveries': 0}
    event_id = get_next_id('ops_equipment_event', 'equipment_event_id', engine)
    events = failures = recoveries = 0
    with transaction(engine) as conn:
        for _, row in states.iterrows():
            equipment_id = int(row['equipment_id'])
            current = row['status']
            if current == 'Available' and rng.random() < equipment_failure_rate:
                new_status, severity, event_type = 'Unavailable', 'Critical', 'Failure'
                failures += 1
            else:
                options = _EQUIPMENT_TRANSITIONS.get(current, [])
                new_status = current
                for candidate, prob in options:
                    if rng.random() < prob:
                        new_status = candidate
                        break
                if new_status == current:
                    continue
                severity = 'Warning' if new_status in ('Warning', 'Unavailable') else 'Info'
                event_type = 'Status Change'
                if current in ('Unavailable', 'Maintenance') and new_status == 'Available':
                    recoveries += 1
            availability = 'Available' if new_status in ('Available', 'InUse') else 'Unavailable'
            utilization = 'InUse' if new_status == 'InUse' else 'Idle'
            upsert_row(conn, 'ops_equipment_state', ['equipment_id'], {
                'equipment_id': equipment_id, 'status': new_status, 'availability_status': availability,
                'utilization_status': utilization, 'battery_pct': None, 'temperature': None, 'pressure': None,
                'error_code': 'ERR-001' if new_status == 'Unavailable' else None, 'maintenance_due_date': None,
                'last_seen_datetime': simulated_now, 'updated_datetime': simulated_now, 'simulation_run_id': run_id,
            })
            insert_rows(conn, 'ops_equipment_event', [{
                'equipment_event_id': event_id, 'equipment_id': equipment_id, 'event_datetime': simulated_now,
                'event_type': event_type, 'severity': severity, 'metric_name': None, 'metric_value': None, 'metric_unit': None,
                'status_before': current, 'status_after': new_status,
                'description': f"Simulated {event_type.lower()} for {row['equipment_type']}", 'simulation_run_id': run_id,
            }])
            event_id += 1
            events += 1
    return {'events': events, 'failures': failures, 'recoveries': recoveries}


In [ ]:
# ============================================================================
# CELL: Alert Engine (opens / escalates / resolves ops_operational_alert)
# ============================================================================

def _get_open_alert(conn, alert_key):
    df = query_db("SELECT * FROM dbo.ops_operational_alert WHERE alert_key = :k AND status <> 'Resolved'", conn, {'k': alert_key})
    return df.iloc[0] if not df.empty else None


def _severity_rank(sev):
    return {'Info': 0, 'Warning': 1, 'Critical': 2}.get(sev, 0)


def _open_or_escalate(conn, alert_id_holder, alert_key, hospital_id, category, alert_type, severity, title,
                       description, source_metric, source_value, threshold_value, simulated_now, run_id, dims):
    existing = _get_open_alert(conn, alert_key)
    if existing is None:
        alert_id = alert_id_holder[0]
        alert_id_holder[0] += 1
        conn.execute(text(
            "INSERT INTO dbo.ops_operational_alert (alert_id, alert_key, hospital_id, building_id, floor_number, "
            "unit_id, room_id, bed_id, equipment_id, encounter_id, alert_category, alert_type, severity, title, "
            "description, status, opened_datetime, source_metric, source_value, threshold_value, simulation_run_id, "
            "created_datetime, updated_datetime) VALUES (:alert_id, :alert_key, :hospital_id, :building_id, "
            ":floor_number, :unit_id, :room_id, :bed_id, :equipment_id, :encounter_id, :category, :alert_type, "
            ":severity, :title, :description, 'Open', :opened, :source_metric, :source_value, :threshold_value, "
            ":run_id, :now, :now)"
        ), {
            'alert_id': alert_id, 'alert_key': alert_key, 'hospital_id': hospital_id, 'building_id': dims.get('building_id'),
            'floor_number': dims.get('floor_number'), 'unit_id': dims.get('unit_id'), 'room_id': dims.get('room_id'),
            'bed_id': dims.get('bed_id'), 'equipment_id': dims.get('equipment_id'), 'encounter_id': dims.get('encounter_id'),
            'category': category, 'alert_type': alert_type, 'severity': severity, 'title': title, 'description': description,
            'opened': simulated_now, 'source_metric': source_metric, 'source_value': source_value,
            'threshold_value': threshold_value, 'run_id': run_id, 'now': simulated_now,
        })
        event_id = get_next_id('ops_alert_event', 'alert_event_id', conn)
        insert_rows(conn, 'ops_alert_event', [{
            'alert_event_id': event_id, 'alert_id': alert_id, 'event_datetime': simulated_now, 'event_type': 'Opened',
            'status_before': None, 'status_after': 'Open', 'description': description, 'simulation_run_id': run_id,
        }])
        return 'opened'
    if existing['severity'] != severity and _severity_rank(severity) > _severity_rank(existing['severity']):
        conn.execute(text(
            "UPDATE dbo.ops_operational_alert SET severity = :sev, source_value = :sv, updated_datetime = :now WHERE alert_id = :aid"
        ), {'sev': severity, 'sv': source_value, 'now': simulated_now, 'aid': int(existing['alert_id'])})
        event_id = get_next_id('ops_alert_event', 'alert_event_id', conn)
        insert_rows(conn, 'ops_alert_event', [{
            'alert_event_id': event_id, 'alert_id': int(existing['alert_id']), 'event_datetime': simulated_now,
            'event_type': 'Escalated', 'status_before': existing['severity'], 'status_after': severity,
            'description': f'Escalated to {severity}', 'simulation_run_id': run_id,
        }])
        return 'escalated'
    return 'unchanged'


def _resolve_if_open(conn, alert_key, simulated_now, run_id):
    existing = _get_open_alert(conn, alert_key)
    if existing is None:
        return False
    conn.execute(text(
        "UPDATE dbo.ops_operational_alert SET status = 'Resolved', resolved_datetime = :now, updated_datetime = :now WHERE alert_id = :aid"
    ), {'now': simulated_now, 'aid': int(existing['alert_id'])})
    event_id = get_next_id('ops_alert_event', 'alert_event_id', conn)
    insert_rows(conn, 'ops_alert_event', [{
        'alert_event_id': event_id, 'alert_id': int(existing['alert_id']), 'event_datetime': simulated_now,
        'event_type': 'Resolved', 'status_before': existing['status'], 'status_after': 'Resolved',
        'description': 'Condition cleared', 'simulation_run_id': run_id,
    }])
    return True


def evaluate_alerts(engine, simulated_now, run_id, scenario):
    opened = escalated = resolved = 0
    alert_id_holder = [get_next_id('ops_operational_alert', 'alert_id', engine)]
    hospitals = query_db('SELECT hospital_id, name FROM dbo.hospitals', engine)
    with transaction(engine) as conn:
        for _, h in hospitals.iterrows():
            hid = int(h['hospital_id'])
            for unit_type, label, threshold_attr in (
                ('Critical Care', 'ICU', 'icu_occupancy_alert_threshold'),
                ('Emergency', 'ED', 'ed_occupancy_alert_threshold'),
            ):
                occ_df = query_db(
                    "SELECT COUNT(*) AS total, SUM(CASE WHEN bs.occupancy_status = 'Occupied' THEN 1 ELSE 0 END) AS occ "
                    "FROM dbo.beds b JOIN dbo.ops_unit u ON u.hospital_id = b.hospital_id AND u.source_floor_number = b.floor_number "
                    "LEFT JOIN dbo.ops_bed_state bs ON b.bed_id = bs.bed_id "
                    "WHERE b.hospital_id = :h AND u.unit_type = :ut", conn, {'h': hid, 'ut': unit_type})
                total = int(occ_df.iloc[0]['total'] or 0) if not occ_df.empty else 0
                occ = int(occ_df.iloc[0]['occ'] or 0) if not occ_df.empty else 0
                pct = (occ / total) if total > 0 else 0.0
                threshold = getattr(scenario, threshold_attr)
                key = f'CAPACITY:{label}_OCCUPANCY:hospital={hid}'
                if total > 0 and pct >= threshold:
                    sev = 'Critical' if pct >= 0.97 else 'Warning'
                    result = _open_or_escalate(conn, alert_id_holder, key, hid, 'Capacity', f'{label} Occupancy High', sev,
                                                f'{label} occupancy above threshold at {h["name"]}',
                                                f'{label} occupancy is {pct:.0%} (threshold {threshold:.0%}).', 'occupancy_pct',
                                                round(pct * 100, 1), round(threshold * 100, 1), simulated_now, run_id, {'unit_id': None})
                    opened += result == 'opened'; escalated += result == 'escalated'
                    if total > 0 and occ >= total:
                        no_bed_key = f'CAPACITY:NO_AVAILABLE_{label}_BEDS:hospital={hid}'
                        result2 = _open_or_escalate(conn, alert_id_holder, no_bed_key, hid, 'Capacity', f'No Available {label} Beds',
                                                     'Critical', f'No available {label} beds at {h["name"]}',
                                                     f'All {total} {label} beds are occupied.', 'available_beds', 0, 0,
                                                     simulated_now, run_id, {})
                        opened += result2 == 'opened'
                    else:
                        resolved += _resolve_if_open(conn, f'CAPACITY:NO_AVAILABLE_{label}_BEDS:hospital={hid}', simulated_now, run_id)
                else:
                    resolved += _resolve_if_open(conn, key, simulated_now, run_id)
                    resolved += _resolve_if_open(conn, f'CAPACITY:NO_AVAILABLE_{label}_BEDS:hospital={hid}', simulated_now, run_id)

            staffing = query_db(
                "SELECT ss.unit_id, ss.staffing_coverage_pct, ss.charge_nurse_staff_id FROM dbo.ops_staffing_state ss "
                "JOIN dbo.ops_unit u ON ss.unit_id = u.unit_id WHERE u.hospital_id = :h", conn, {'h': hid})
            for _, s in staffing.iterrows():
                unit_id = int(s['unit_id'])
                cov = float(s['staffing_coverage_pct'] or 100.0) / 100.0
                key = f'STAFFING:RN_COVERAGE_LOW:unit={unit_id}'
                if cov < scenario.rn_coverage_alert_threshold:
                    sev = 'Critical' if cov < 0.6 else 'Warning'
                    result = _open_or_escalate(conn, alert_id_holder, key, hid, 'Staffing', 'RN Coverage Below Target', sev,
                                                f'RN coverage below target on unit {unit_id}', f'Coverage is {cov:.0%}.',
                                                'staffing_coverage_pct', round(cov * 100, 1),
                                                round(scenario.rn_coverage_alert_threshold * 100, 1), simulated_now, run_id, {'unit_id': unit_id})
                    opened += result == 'opened'; escalated += result == 'escalated'
                else:
                    resolved += _resolve_if_open(conn, key, simulated_now, run_id)
                cn_key = f'STAFFING:OPEN_CHARGE_NURSE:unit={unit_id}'
                if pd.isna(s['charge_nurse_staff_id']):
                    result = _open_or_escalate(conn, alert_id_holder, cn_key, hid, 'Staffing', 'Open Charge Nurse Assignment',
                                                'Warning', f'No charge nurse assigned on unit {unit_id}',
                                                'Charge nurse slot is open for the current shift.', None, None, None,
                                                simulated_now, run_id, {'unit_id': unit_id})
                    opened += result == 'opened'
                else:
                    resolved += _resolve_if_open(conn, cn_key, simulated_now, run_id)

            delayed = query_db(
                "SELECT COUNT(*) AS n FROM dbo.ops_discharge_readiness dr JOIN dbo.admissions a ON dr.encounter_id = a.encounter_id "
                "WHERE a.hospital_id = :h AND dr.readiness_status = 'Not Ready' AND dr.outstanding_barrier_count >= 2",
                conn, {'h': hid})
            delay_n = int(delayed.iloc[0]['n']) if not delayed.empty else 0
            key = f'PATIENT_FLOW:DISCHARGE_DELAYS:hospital={hid}'
            if delay_n >= 5:
                result = _open_or_escalate(conn, alert_id_holder, key, hid, 'Patient Flow', 'Excessive Discharge Delays',
                                            'Critical' if delay_n >= 15 else 'Warning', f'Excessive discharge delays at {h["name"]}',
                                            f'{delay_n} encounters have >=2 outstanding discharge barriers.', 'delayed_discharge_count',
                                            delay_n, 5, simulated_now, run_id, {})
                opened += result == 'opened'; escalated += result == 'escalated'
            else:
                resolved += _resolve_if_open(conn, key, simulated_now, run_id)

            for equip_type, label in (('Ventilator', 'Ventilator'), ('MRI', 'Imaging'), ('CT Scanner', 'Imaging'), ('Portable X-Ray', 'Imaging')):
                unavailable = query_db(
                    "SELECT eq.equipment_id FROM dbo.ops_equipment eq JOIN dbo.ops_equipment_state es ON eq.equipment_id = es.equipment_id "
                    "WHERE eq.hospital_id = :h AND eq.equipment_type = :t AND es.availability_status = 'Unavailable'",
                    conn, {'h': hid, 't': equip_type})
                for _, eq_row in unavailable.iterrows():
                    key = f'EQUIPMENT:{label.upper()}_UNAVAILABLE:equipment={int(eq_row["equipment_id"])}'
                    result = _open_or_escalate(conn, alert_id_holder, key, hid, 'Equipment', f'{equip_type} Unavailable',
                                                'Critical' if equip_type == 'Ventilator' else 'Warning',
                                                f'{equip_type} unavailable at {h["name"]}',
                                                f'{equip_type} (id {int(eq_row["equipment_id"])}) is currently unavailable.', None, None, None,
                                                simulated_now, run_id, {'equipment_id': int(eq_row['equipment_id'])})
                    opened += result == 'opened'; escalated += result == 'escalated'

            blocked = query_db(
                "SELECT rs.room_id FROM dbo.ops_room_state rs JOIN dbo.rooms r ON rs.room_id = r.room_id "
                "WHERE r.hospital_id = :h AND rs.operational_status IN ('Maintenance', 'Closed')", conn, {'h': hid})
            for _, r_row in blocked.iterrows():
                key = f'ENVIRONMENTAL:ROOM_BLOCKED:room={int(r_row["room_id"])}'
                result = _open_or_escalate(conn, alert_id_holder, key, hid, 'Environmental', 'Room Blocked For Maintenance',
                                            'Warning', f'Room {int(r_row["room_id"])} blocked for maintenance',
                                            'Room is out of service.', None, None, None, simulated_now, run_id,
                                            {'room_id': int(r_row['room_id'])})
                opened += result == 'opened'
    return {'opened': opened, 'escalated': escalated, 'resolved': resolved}


In [ ]:
# ============================================================================
# CELL: Simulation Control / Run / Checkpoint bookkeeping
# ============================================================================
STALE_HEARTBEAT_MINUTES = 5


def ensure_control_row(engine, simulator_name, scenario_name, update_interval_seconds, speed_multiplier, random_seed, selected_hospital_id):
    existing = query_db('SELECT * FROM dbo.ops_simulation_control WHERE simulator_name = :n', engine, {'n': simulator_name})
    if existing.empty:
        control_id = get_next_id('ops_simulation_control', 'control_id', engine)
        with transaction(engine) as conn:
            upsert_row(conn, 'ops_simulation_control', ['control_id'], {
                'control_id': control_id, 'simulator_name': simulator_name, 'requested_state': 'RUN',
                'update_interval_seconds': update_interval_seconds, 'speed_multiplier': speed_multiplier,
                'random_seed': random_seed, 'scenario_name': scenario_name, 'selected_hospital_id': selected_hospital_id,
                'last_updated_datetime': pd.Timestamp.utcnow().tz_localize(None), 'updated_by': 'setup',
            })
        existing = query_db('SELECT * FROM dbo.ops_simulation_control WHERE simulator_name = :n', engine, {'n': simulator_name})
    return existing.iloc[0].to_dict()


def get_control_state(engine, simulator_name):
    df = query_db('SELECT * FROM dbo.ops_simulation_control WHERE simulator_name = :n', engine, {'n': simulator_name})
    return df.iloc[0].to_dict() if not df.empty else None


def set_requested_state(engine, simulator_name, requested_state, updated_by='notebook'):
    assert requested_state in ('RUN', 'PAUSE', 'STOP')
    with transaction(engine) as conn:
        conn.execute(text(
            "UPDATE dbo.ops_simulation_control SET requested_state = :s, last_updated_datetime = SYSUTCDATETIME(), "
            "updated_by = :u WHERE simulator_name = :n"
        ), {'s': requested_state, 'u': updated_by, 'n': simulator_name})


def find_resumable_run(engine, simulator_name, stale_minutes=STALE_HEARTBEAT_MINUTES):
    df = query_db(
        "SELECT TOP 1 * FROM dbo.ops_simulation_run WHERE simulator_name = :n AND current_status = 'RUNNING' "
        "ORDER BY simulation_run_id DESC", engine, {'n': simulator_name})
    if df.empty:
        return None
    row = df.iloc[0]
    last_hb = row['last_successful_iteration_datetime'] or row['started_datetime']
    age_minutes = (pd.Timestamp.utcnow().tz_localize(None) - pd.Timestamp(last_hb)).total_seconds() / 60.0
    return row.to_dict() if age_minutes >= stale_minutes else None


def start_run(engine, simulator_name, scenario_name, random_seed, config):
    run_id = get_next_id('ops_simulation_run', 'simulation_run_id', engine)
    now = pd.Timestamp.utcnow().tz_localize(None)
    with transaction(engine) as conn:
        upsert_row(conn, 'ops_simulation_run', ['simulation_run_id'], {
            'simulation_run_id': run_id, 'simulator_name': simulator_name, 'scenario_name': scenario_name,
            'random_seed': random_seed, 'started_datetime': now, 'ended_datetime': None, 'current_status': 'RUNNING',
            'iteration_count': 0, 'last_successful_iteration_datetime': now, 'error_count': 0,
            'hostname': socket.gethostname(), 'configuration_json': json.dumps(config, default=str),
        })
    return run_id


def update_heartbeat(engine, run_id, iteration_count, error_count=0):
    with transaction(engine) as conn:
        conn.execute(text(
            "UPDATE dbo.ops_simulation_run SET iteration_count = :it, error_count = :err, "
            "last_successful_iteration_datetime = SYSUTCDATETIME() WHERE simulation_run_id = :rid"
        ), {'it': iteration_count, 'err': error_count, 'rid': run_id})


def complete_run(engine, run_id, status):
    with transaction(engine) as conn:
        conn.execute(text(
            "UPDATE dbo.ops_simulation_run SET current_status = :s, ended_datetime = SYSUTCDATETIME() WHERE simulation_run_id = :rid"
        ), {'s': status, 'rid': run_id})


def save_checkpoint(engine, simulator_name, run_id, iteration, simulated_datetime, random_state_json=None, extra=None):
    with transaction(engine) as conn:
        upsert_row(conn, 'ops_simulation_checkpoint', ['simulator_name', 'simulation_run_id'], {
            'simulator_name': simulator_name, 'simulation_run_id': run_id, 'last_completed_iteration': iteration,
            'simulated_datetime': simulated_datetime, 'random_state': random_state_json,
            'checkpoint_datetime': pd.Timestamp.utcnow().tz_localize(None), 'checkpoint_json': json.dumps(extra or {}, default=str),
        })


def load_latest_checkpoint(engine, simulator_name, run_id):
    df = query_db('SELECT * FROM dbo.ops_simulation_checkpoint WHERE simulator_name = :n AND simulation_run_id = :rid',
                   engine, {'n': simulator_name, 'rid': run_id})
    return df.iloc[0].to_dict() if not df.empty else None


def log_event(engine, run_id, iteration_number, category, action, status='OK', message=None, entity_type=None, entity_id=None):
    event_id = get_next_id('ops_simulation_event_log', 'simulation_event_log_id', engine)
    if message is not None and len(message) > 1000:
        message = message[:997] + '...'
    with transaction(engine) as conn:
        conn.execute(text(
            "INSERT INTO dbo.ops_simulation_event_log (simulation_event_log_id, simulation_run_id, iteration_number, "
            "event_datetime, event_category, entity_type, entity_id, action, status, message) VALUES "
            "(:id, :rid, :it, SYSUTCDATETIME(), :cat, :etype, :eid, :action, :status, :msg)"
        ), {'id': event_id, 'rid': run_id, 'it': iteration_number, 'cat': category, 'etype': entity_type,
            'eid': entity_id, 'action': action, 'status': status, 'msg': message})


In [ ]:
# ============================================================================
# CELL: PASS/WARN/FAIL Validation Checks
# ============================================================================

def _rowcount_check(engine, name, sql, warn_only=False):
    # A query that cannot run is reported as a failure, never silently passed,
    # but it must not abort the remaining checks in a notebook run.
    try:
        df = pd.read_sql(text(sql), engine)
    except Exception as exc:
        return {'check': name, 'status': 'FAIL', 'detail': f'check could not run: {exc}'}
    n = len(df)
    if n == 0:
        return {'check': name, 'status': 'PASS', 'detail': '0 violations'}
    return {'check': name, 'status': 'WARN' if warn_only else 'FAIL', 'detail': f'{n} row(s) violate this check'}


def run_all_checks(engine=None):
    engine = engine or SINGLE_ENGINE
    checks = [
        _rowcount_check(engine, 'Every ops_building references a valid hospital', '''
            SELECT b.building_id FROM dbo.ops_building b
            LEFT JOIN dbo.hospitals h ON b.hospital_id = h.hospital_id WHERE h.hospital_id IS NULL'''),
        _rowcount_check(engine, 'Every ops_unit references a valid building/hospital/floors row', '''
            SELECT u.unit_id FROM dbo.ops_unit u
            LEFT JOIN dbo.ops_building b ON u.building_id = b.building_id
            LEFT JOIN dbo.hospitals h ON u.hospital_id = h.hospital_id
            LEFT JOIN dbo.floors f ON f.hospital_id = u.hospital_id AND f.floor_number = u.source_floor_number
            WHERE b.building_id IS NULL OR h.hospital_id IS NULL OR f.floor_number IS NULL'''),
        _rowcount_check(engine, 'Every ops_room_attribute references a valid room and unit', '''
            SELECT ra.room_id FROM dbo.ops_room_attribute ra
            LEFT JOIN dbo.rooms r ON ra.room_id = r.room_id
            LEFT JOIN dbo.ops_unit u ON ra.unit_id = u.unit_id
            WHERE r.room_id IS NULL OR u.unit_id IS NULL'''),
        _rowcount_check(engine, 'Every ops_bed_state references a valid bed', '''
            SELECT bs.bed_id FROM dbo.ops_bed_state bs
            LEFT JOIN dbo.beds b ON bs.bed_id = b.bed_id WHERE b.bed_id IS NULL'''),
        _rowcount_check(engine, 'ops_unit.staffed_bed_count matches actual bed count on that floor', '''
            SELECT u.unit_id FROM dbo.ops_unit u
            WHERE u.staffed_bed_count <> (
                SELECT COUNT(*) FROM dbo.beds b WHERE b.hospital_id = u.hospital_id AND b.floor_number = u.source_floor_number)'''),
        _rowcount_check(engine, 'No bed has more than one active (Occupied) encounter', '''
            SELECT bed_id FROM dbo.ops_bed_state WHERE occupancy_status = 'Occupied' GROUP BY bed_id HAVING COUNT(*) > 1'''),
        _rowcount_check(engine, 'No encounter occupies more than one bed at a time', '''
            SELECT encounter_id FROM dbo.ops_bed_state WHERE occupancy_status = 'Occupied' AND encounter_id IS NOT NULL
            GROUP BY encounter_id HAVING COUNT(DISTINCT bed_id) > 1'''),
        _rowcount_check(engine, 'Occupied beds have patient+encounter; available/cleaning beds do not', '''
            SELECT bed_id FROM dbo.ops_bed_state
            WHERE (occupancy_status = 'Occupied' AND (encounter_id IS NULL OR patient_id IS NULL))
               OR (occupancy_status IN ('Available', 'Cleaning') AND encounter_id IS NOT NULL)'''),
        _rowcount_check(engine, 'Charge nurses reference valid ops_staff rows', '''
            SELECT ss.unit_id FROM dbo.ops_staffing_state ss
            LEFT JOIN dbo.ops_staff s ON ss.charge_nurse_staff_id = s.staff_id
            WHERE ss.charge_nurse_staff_id IS NOT NULL AND s.staff_id IS NULL'''),
        _rowcount_check(engine, 'Equipment locations are valid (unit hospital matches equipment hospital)', '''
            SELECT eq.equipment_id FROM dbo.ops_equipment eq
            LEFT JOIN dbo.ops_unit u ON eq.unit_id = u.unit_id
            WHERE u.unit_id IS NULL OR u.hospital_id <> eq.hospital_id'''),
        _rowcount_check(engine, 'Active alerts reference a valid hospital', '''
            SELECT alert_id FROM dbo.ops_operational_alert oa
            LEFT JOIN dbo.hospitals h ON oa.hospital_id = h.hospital_id
            WHERE oa.status <> 'Resolved' AND h.hospital_id IS NULL'''),
        _rowcount_check(engine, 'Hospital snapshot occupancy_pct matches occupied/staffed beds', '''
            SELECT hospital_id FROM dbo.vw_ops_hospital_snapshot
            WHERE ABS(occupancy_pct - CASE WHEN staffed_beds > 0 THEN ROUND(100.0 * occupied_beds / staffed_beds, 1) ELSE 0 END) > 0.1'''),
        _rowcount_check(
            engine, 'ops_unit staffed-bed sum vs latest hospital_department_beds.beds_allocated sum (informational)', '''
            SELECT h.hospital_id FROM dbo.hospitals h
            WHERE ISNULL((SELECT SUM(staffed_bed_count) FROM dbo.ops_unit WHERE hospital_id = h.hospital_id), 0) <>
                  ISNULL((SELECT SUM(hdb.beds_allocated) FROM dbo.hospital_department_beds hdb
                          WHERE hdb.hospital_id = h.hospital_id AND hdb.date = (SELECT MAX(date) FROM dbo.hospital_department_beds)), 0)''',
            warn_only=True),
        _rowcount_check(engine, 'Simulated episode has matching admission and patient', '''
            SELECT se.encounter_id FROM dbo.ops_simulated_episode se
            LEFT JOIN dbo.admissions a ON a.admission_id = se.admission_id
            LEFT JOIN dbo.encounters e ON e.encounter_id = se.encounter_id
            WHERE a.encounter_id IS NULL OR a.encounter_id <> se.encounter_id
               OR a.patient_id <> se.patient_id OR e.patient_id <> se.patient_id'''),
        _rowcount_check(engine, 'Simulated episode has matching diagnosis, lab and medication', '''
            SELECT se.encounter_id FROM dbo.ops_simulated_episode se
            WHERE NOT EXISTS (SELECT 1 FROM dbo.diagnoses d WHERE d.encounter_id = se.encounter_id
                AND d.patient_id = se.patient_id AND d.onset_date >= CAST(se.created_datetime AS DATE))
               OR NOT EXISTS (SELECT 1 FROM dbo.labs l WHERE l.encounter_id = se.encounter_id
                AND l.patient_id = se.patient_id AND l.order_date >= CAST(se.created_datetime AS DATE)
                AND l.reference_range <> 'Normal')
               OR NOT EXISTS (SELECT 1 FROM dbo.medications m WHERE m.encounter_id = se.encounter_id
                AND m.patient_id = se.patient_id AND m.start_date >= CAST(se.created_datetime AS DATE))'''),
        _rowcount_check(engine, 'Simulated movement links and chronology are valid', '''
            SELECT pm.movement_id FROM dbo.ops_patient_movement pm
            JOIN dbo.ops_simulated_episode se ON se.encounter_id = pm.encounter_id
            WHERE pm.patient_id <> se.patient_id
               OR pm.completed_datetime IS NULL OR pm.requested_datetime > pm.completed_datetime
               OR (pm.movement_type IN ('Internal Transfer', 'ICU Transfer')
                   AND (pm.from_unit_id IS NULL OR pm.to_unit_id IS NULL))
               OR (pm.movement_type = 'ICU Transfer' AND NOT EXISTS (
                   SELECT 1 FROM dbo.ops_unit u WHERE u.unit_id = pm.to_unit_id
                     AND u.unit_type = 'Critical Care'))'''),
        _rowcount_check(engine, 'Simulated discharge agrees with clinical admission', '''
            SELECT se.encounter_id FROM dbo.ops_simulated_episode se
            JOIN dbo.admissions a ON a.admission_id = se.admission_id
            WHERE EXISTS (SELECT 1 FROM dbo.ops_patient_movement pm
                WHERE pm.encounter_id = se.encounter_id AND pm.movement_type = 'Discharge'
                  AND pm.movement_status = 'Completed'
                  AND (a.discharge_datetime IS NULL OR a.discharge_datetime < a.admit_datetime
                       OR DATEDIFF(SECOND, a.discharge_datetime, pm.completed_datetime) <> 0))
               OR (a.discharge_datetime IS NOT NULL AND NOT EXISTS (
                   SELECT 1 FROM dbo.ops_patient_movement pm
                   WHERE pm.encounter_id = se.encounter_id AND pm.movement_type = 'Discharge'
                     AND pm.movement_status = 'Completed'))'''),
    ]
    return pd.DataFrame(checks)


def print_validation_summary(results):
    fail = int((results['status'] == 'FAIL').sum())
    warn = int((results['status'] == 'WARN').sum())
    passed = int((results['status'] == 'PASS').sum())
    overall = 'FAIL' if fail else ('WARN' if warn else 'PASS')
    print(f'Validation summary: {passed} PASS, {warn} WARN, {fail} FAIL -> overall {overall}')
    for _, row in results.iterrows():
        marker = {'PASS': '\u2713', 'WARN': '\u26a0', 'FAIL': '\u2717'}[row['status']]
        print(f"  {marker} [{row['status']}] {row['check']} -- {row['detail']}")
    return overall


In [ ]:
# ============================================================================
# CELL: Application Snapshot Refresh
# All vw_ops_* objects are plain SQL VIEWs computed live from current-state
# tables -- "refreshing" only means keeping room aggregates up to date.
# ============================================================================

def refresh_snapshots(engine=None, run_id=None):
    refresh_room_aggregates(engine or SINGLE_ENGINE, run_id)


def get_health_system_summary(engine=None):
    df = query_db('SELECT * FROM dbo.vw_ops_health_system_snapshot', engine or SINGLE_ENGINE)
    return df.iloc[0].to_dict() if not df.empty else {}


def get_hospital_summary(engine=None, hospital_id=None):
    engine = engine or SINGLE_ENGINE
    if hospital_id:
        return query_db('SELECT * FROM dbo.vw_ops_hospital_snapshot WHERE hospital_id = :h', engine, {'h': hospital_id})
    return query_db('SELECT * FROM dbo.vw_ops_hospital_snapshot', engine)


In [ ]:
# ============================================================================
# CELL: Real-Time Simulation Loop
# Reads ops_simulation_control every iteration (RUN/PAUSE/STOP). Simulated
# time = simulated_start + (wall_elapsed * SPEED_MULTIPLIER). A
# KeyboardInterrupt (stopping this cell) triggers a graceful shutdown:
# final checkpoint + run marked STOPPED_BY_USER.
# ============================================================================

def _simulated_now(sim_time_base, wall_start, speed_multiplier):
    wall_elapsed_seconds = time.monotonic() - wall_start
    return sim_time_base + pd.Timedelta(seconds=wall_elapsed_seconds * speed_multiplier)


def reset_current_state(engine=None):
    '''Optional demo helper (RESET_CURRENT_STATE=True): clears current-state
    tables only (never the append-only event tables or reference/hierarchy
    tables), then re-seeds bed/room state from patient_bed_assignments.'''
    engine = engine or SINGLE_ENGINE
    print('RESET_CURRENT_STATE=True -- clearing current-state tables...')
    with transaction(engine) as conn:
        for t in ('ops_operational_alert', 'ops_discharge_readiness', 'ops_staffing_state',
                  'ops_equipment_state', 'ops_room_state', 'ops_bed_state'):
            conn.execute(text(f'DELETE FROM dbo.{t}'))
    initialize_bed_state(engine)
    initialize_room_state(engine)
    initialize_equipment_state(engine)
    print('  \u2713 current-state tables reset and re-seeded.')


def run_one_iteration(engine, run_id, iteration_number, simulated_now, scenario, rng):
    metrics = {'admitted': 0, 'discharge_progressed': 0, 'discharged': 0, 'transferred': 0,
               'rooms_cleaned': 0, 'equipment_events': 0, 'equipment_failures': 0, 'equipment_recoveries': 0,
               'alerts_opened': 0, 'alerts_escalated': 0, 'alerts_resolved': 0}

    if ENABLE_PATIENT_FLOW:
        metrics.update(simulate_patient_flow_iteration(engine, simulated_now, run_id, scenario, BATCH_SIZE, rng))
        metrics['rooms_cleaned'] = simulate_room_cleaning_iteration(engine, run_id, rng)

    if ENABLE_STAFFING_EVENTS:
        compute_staffing_state(engine, run_id, staffing_absence_rate=scenario.staffing_absence_rate)

    if ENABLE_EQUIPMENT_EVENTS:
        eq_metrics = simulate_equipment_iteration(engine, simulated_now, run_id, scenario.equipment_failure_rate, BATCH_SIZE, rng)
        metrics['equipment_events'] = eq_metrics['events']
        metrics['equipment_failures'] = eq_metrics['failures']
        metrics['equipment_recoveries'] = eq_metrics['recoveries']

    if ENABLE_ALERTS:
        alert_metrics = evaluate_alerts(engine, simulated_now, run_id, scenario)
        metrics['alerts_opened'] = alert_metrics['opened']
        metrics['alerts_escalated'] = alert_metrics['escalated']
        metrics['alerts_resolved'] = alert_metrics['resolved']

    if ENABLE_SNAPSHOT_REFRESH:
        refresh_snapshots(engine, run_id)

    return metrics


def run_loop(engine=None):
    engine = engine or SINGLE_ENGINE
    ensure_control_row(engine, SIMULATOR_NAME, SCENARIO_NAME, UPDATE_INTERVAL_SECONDS, SPEED_MULTIPLIER, RANDOM_SEED, SELECTED_HOSPITAL_ID)
    # Starting this cell always means "run" -- force past any stale PAUSE/STOP left over
    # from a previous session (e.g. someone stopped an earlier run and never reset it).
    set_requested_state(engine, SIMULATOR_NAME, 'RUN', updated_by='run_loop_start')

    if RESET_CURRENT_STATE:
        reset_current_state(engine)

    resumable = find_resumable_run(engine, SIMULATOR_NAME) if RESUME_FROM_CHECKPOINT else None
    if resumable is not None:
        run_id = int(resumable['simulation_run_id'])
        cp = load_latest_checkpoint(engine, SIMULATOR_NAME, run_id)
        start_iteration = int(cp['last_completed_iteration']) + 1 if cp else 1
        sim_time_base = pd.Timestamp(cp['simulated_datetime']) if cp and cp.get('simulated_datetime') is not None else pd.Timestamp.utcnow().tz_localize(None)
        print(f'Resuming run {run_id} from iteration {start_iteration} (simulated time {sim_time_base}).')
    else:
        run_id = start_run(engine, SIMULATOR_NAME, SCENARIO_NAME, RANDOM_SEED, {
            'update_interval_seconds': UPDATE_INTERVAL_SECONDS, 'speed_multiplier': SPEED_MULTIPLIER,
            'batch_size': BATCH_SIZE, 'selected_hospital_id': SELECTED_HOSPITAL_ID,
        })
        start_iteration = 1
        sim_time_base = pd.Timestamp.utcnow().tz_localize(None)
        print(f'Started new run {run_id} (scenario={SCENARIO_NAME}).')

    scenario = get_scenario_profile(SCENARIO_NAME)
    rng = random.Random(RANDOM_SEED)
    wall_start = time.monotonic()
    loop_start = time.monotonic()
    iteration = start_iteration
    consecutive_errors = 0
    totals = {k: 0 for k in ['admitted', 'discharged', 'transferred', 'alerts_opened', 'alerts_resolved']}
    final_status = 'COMPLETED'

    try:
        while True:
            control = get_control_state(engine, SIMULATOR_NAME) or {}
            requested_state = control.get('requested_state', 'RUN')

            if requested_state == 'STOP':
                print('STOP requested via ops_simulation_control -- exiting cleanly.')
                final_status = 'STOPPED_BY_USER'
                break
            if requested_state == 'PAUSE':
                update_heartbeat(engine, run_id, iteration - 1, consecutive_errors)
                print(heartbeat_line(run_id=run_id, state='PAUSED', iteration=iteration))
                time.sleep(UPDATE_INTERVAL_SECONDS)
                continue

            if MAX_ITERATIONS and iteration > MAX_ITERATIONS:
                print(f'Reached MAX_ITERATIONS={MAX_ITERATIONS} -- stopping.')
                break
            if MAX_RUNTIME_MINUTES and (time.monotonic() - loop_start) / 60.0 >= MAX_RUNTIME_MINUTES:
                print(f'Reached MAX_RUNTIME_MINUTES={MAX_RUNTIME_MINUTES} -- stopping.')
                break

            simulated_now = _simulated_now(sim_time_base, wall_start, SPEED_MULTIPLIER)
            iter_start = time.monotonic()
            try:
                metrics = run_one_iteration(engine, run_id, iteration, simulated_now, scenario, rng)
                consecutive_errors = 0
            except Exception as exc:
                consecutive_errors += 1
                print(f'Iteration {iteration} failed ({consecutive_errors}/{MAX_CONSECUTIVE_ERRORS}): {exc}')
                try:
                    log_event(engine, run_id, iteration, 'Error', 'iteration_failed', status='ERROR', message=str(exc))
                except Exception as log_exc:
                    print(f'  (also failed to write the error to ops_simulation_event_log: {log_exc})')
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    print('Max consecutive errors reached -- stopping.')
                    final_status = 'ERROR'
                    break
                time.sleep(min(30, 2 ** consecutive_errors))
                continue

            for k in totals:
                totals[k] += metrics.get(k, 0)

            write_ms = round((time.monotonic() - iter_start) * 1000, 1)
            update_heartbeat(engine, run_id, iteration, consecutive_errors)
            if iteration % CHECKPOINT_INTERVAL_ITERATIONS == 0:
                save_checkpoint(engine, SIMULATOR_NAME, run_id, iteration, simulated_now)

            print(heartbeat_line(
                run_id=run_id, scenario=SCENARIO_NAME, state='RUN', iteration=iteration,
                sim_time=simulated_now.isoformat(sep=' ', timespec='seconds'),
                admitted=metrics['admitted'], discharged=metrics['discharged'], transferred=metrics['transferred'],
                beds_changed=metrics['rooms_cleaned'], equip_events=metrics['equipment_events'],
                alerts_opened=metrics['alerts_opened'], alerts_resolved=metrics['alerts_resolved'],
                write_ms=write_ms, errors=consecutive_errors,
                next_checkpoint_in=CHECKPOINT_INTERVAL_ITERATIONS - (iteration % CHECKPOINT_INTERVAL_ITERATIONS),
            ))

            iteration += 1
            time.sleep(UPDATE_INTERVAL_SECONDS)
    except KeyboardInterrupt:
        print('KeyboardInterrupt -- shutting down gracefully.')
        final_status = 'STOPPED_BY_USER'

    save_checkpoint(engine, SIMULATOR_NAME, run_id, iteration - 1, _simulated_now(sim_time_base, wall_start, SPEED_MULTIPLIER))
    complete_run(engine, run_id, final_status)
    summary = {'run_id': run_id, 'final_status': final_status, 'iterations_completed': iteration - 1, **totals}
    print(f'\nRun {run_id} finished with status {final_status} after {iteration - 1} iteration(s).')
    return summary


In [ ]:
# ============================================================================
# CELL: Preflight -- confirm the operations schema exists and is up to date
# The DDL embedded in this notebook is idempotent (IF NOT EXISTS), so applying
# it here self-heals an older deployment that predates a newly added table
# (e.g. ops_simulated_episode) without needing to re-run the Setup notebook.
# ============================================================================
existing_ops_tables = list_ops_tables(SINGLE_ENGINE)
if not existing_ops_tables:
    raise RuntimeError('No ops_* tables found -- run Hospital_Operations_Setup.ipynb first.')

print('Applying any additive schema upgrades (idempotent)...')
deploy_schema(SINGLE_ENGINE, verbose=False)

REQUIRED_OPS_TABLES = ['ops_bed_state', 'ops_patient_movement', 'ops_discharge_readiness', 'ops_simulated_episode']
existing_ops_tables = list_ops_tables(SINGLE_ENGINE)
still_missing = [t for t in REQUIRED_OPS_TABLES if t not in existing_ops_tables]
if still_missing:
    raise RuntimeError(
        'Operations schema is incomplete after deployment -- missing: '
        + ', '.join(still_missing)
        + '. Re-run Hospital_Operations_Setup.ipynb.'
    )
print(f'\u2713 Found {len(existing_ops_tables)} ops_* tables. Ready to start the real-time loop.')
print(f'Parameters: scenario={SCENARIO_NAME}, update_interval={UPDATE_INTERVAL_SECONDS}s, speed_multiplier={SPEED_MULTIPLIER}x, '
      f'max_iterations={MAX_ITERATIONS or "unlimited"}, max_runtime_minutes={MAX_RUNTIME_MINUTES or "unlimited"}')


In [ ]:
# ============================================================================
# CELL: Main Loop -- runs until STOP / MAX_ITERATIONS / MAX_RUNTIME_MINUTES /
# manual interrupt. Safe to interrupt this cell at any time.
# ============================================================================
run_summary = run_loop(SINGLE_ENGINE)
run_summary


## Run summary & diagnostics

The cell above prints a compact heartbeat every iteration and returns a final
summary dict (`run_id`, `final_status`, `iterations_completed`, totals for
admitted/discharged/transferred/alerts).

Useful follow-up queries:
```python
query_db("SELECT * FROM dbo.ops_simulation_run ORDER BY simulation_run_id DESC")
query_db("SELECT TOP 50 * FROM dbo.ops_simulation_event_log WHERE simulation_run_id = :rid ORDER BY simulation_event_log_id DESC", SINGLE_ENGINE, {'rid': run_summary['run_id']})
query_db("SELECT * FROM dbo.vw_ops_active_alerts")
query_db("SELECT * FROM dbo.vw_ops_health_system_snapshot")
run_all_checks(SINGLE_ENGINE)  # re-run PASS/WARN/FAIL validation after the loop stops
```

**Pause / resume / stop without restarting the kernel** (run from a new cell
while the loop above is still executing, or from another notebook connected
to the same database):
```python
set_requested_state(SINGLE_ENGINE, SIMULATOR_NAME, 'PAUSE')
set_requested_state(SINGLE_ENGINE, SIMULATOR_NAME, 'RUN')
set_requested_state(SINGLE_ENGINE, SIMULATOR_NAME, 'STOP')
```

**Reset and start clean** (clears current-state only, never event history or
clinical tables): set `RESET_CURRENT_STATE = True` in the config cell and
re-run from the top, or call `reset_current_state(SINGLE_ENGINE)` directly.
